# Eyherabide, Rokem, Herz & Samengo (2008) on CRCNS ia-1

Paper: H. G. Eyherabide, A. Rokem, A. V. M. Herz and I. Samengo (2008), "Burst firing is a neural
code in an insect auditory system", Frontiers in Computational Neuroscience,
[doi:10.3389/neuro.10.003.2008](https://doi.org/10.3389/neuro.10.003.2008). The claims file
(`claims/crcns-ia-1/10.3389-neuro.10.003.2008.yaml`) was written from the version of record in
PubMed Central (PMC2525941), with its Methods and the legends of Figures 1 to 10 and Table 1; the
paper has no supplement and links no code.

Dataset: CRCNS data set ia-1, [doi:10.6080/K0BG2KWB](https://doi.org/10.6080/K0BG2KWB), read from
the public bucket `crcnsarchive` (`ia-1/crcns-ia1.zip`), streamed and opened in place.

Version: CRCNS data sets are not versioned, so the version is recorded as `unversioned` and the
file read is pinned by its MD5 sum in the data set's `checksums.md5`: `crcns-ia1.zip`
f0740514fabc6995fe54b2a361b6c763 (the helper checks the object's ETag against it before reading;
the object was last modified 2026-02-11). This is the only published state of the data set, so it
is the one pinned. Only `Data1` is used: the data descriptor names it as the data of this paper;
`Data2` (2004) is unpublished and used other stimuli.

| Claim | Tested here |
|---|---|
| data-sessions-and-trials | tested |
| table1-sessions-per-protocol | tested |
| firing-rate-uniform | tested |
| threshold-population | blocked: the thresholds are not in the data set |
| table1-bursting-sessions | tested |
| table1-isolated-spikes | tested |
| table1-burst-length | tested |
| fig5-burst-probability-vs-stimulus | tested |
| fig3-correlation-shape | tested (the data part; the model panels need the thresholds) |
| method-correlation-error | tested |
| model-matches-classification | blocked: the model needs the thresholds s_TH |
| fig6-bursts-less-correlated | tested |
| fig7ab-information-per-burst-increases | tested |
| fig7cd-occurrence-decreases | tested |
| fig7ef-jitter-independent-of-n | tested |
| fig7gh-information-rate | tested |
| direct-method-comparison | tested |
| fig8-nbta-differ | tested |
| fig8-latency-decreases | tested |
| fig8e-sigma-minimum | tested |
| sigma-n-below-sta-sigma | tested |
| fig9-nbta-not-concatenated-1bta | tested (data only; the model sessions need the thresholds) |
| fig9c-population-nbta | tested |
| fig10-burst-n-vs-height | tested |
| firing-rate-no-effect-on-limit | tested |
| model-reproduces-coding-trends | blocked: depends on model-matches-classification |

Two analyses draw random numbers, the shifted-onset baseline of Figure 7 and the size-matched
control of sigma-n-below-sta-sigma; each has its own seeded generator.

Revised after the first verification round. The main population is now selection (c), which also
leaves out sessions with fewer than 98 trials and sessions sorted off-line (see "Session
selection"); the first round's population, selection (a), is shown beside it. The central maximum
of the burst-triggered averages is searched before the burst onset; a first-peak reading with the
refractory plateau left out, a shifted-onset baseline for the information measures, a size-matched
control for sigma_n, sensitivity to the minimum number of bursts, and checks of the trial blocks and
of the `firing_rates` file were added.

In [ ]:
import collections
import itertools
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ia1

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("data set", ia1.CRCNS_ID, ia1.VERSION, "doi", ia1.DATASET_DOI, "zip md5", ia1.ZIP_MD5)
z = ia1.open_zip()
sessions = ia1.load_sessions(z)
cells = ia1.load_singlecells(z)
shapes = {name: ia1.load_stimulus_shape(z, name) for name in sorted({s.stimulus for s in sessions})}
cell_of = {e: i for i, row in enumerate(cells) for e in row}

## Data check

What was loaded, and the checks of the description against the values. The data descriptor
says: spike times in microseconds from the stimulus trigger; trials of the first 1000 ms of the
stimulus, 98 to 533 per stimulus; 700 ms pauses; two interleaved stimuli per experiment;
`cell.dat` gives the stimulus names and intensities (the maximal intensity, mean plus 3 SD);
stimulus files at 20 kHz, 10 s long; `singlecells.txt` lists the experiments made on one cell.

In [ ]:
rows = []
for s in sessions:
    r = ia1.trial_rates(s.trials)
    rows.append(dict(session=s.name, exp=s.exp, date=s.exp[:8], cell=cell_of.get(s.exp, -1),
                     stimulus=s.stimulus, protocol=s.protocol, cutoff=s.cutoff, intensity=s.intensity,
                     header_intensity=float(s.header["intensity (dB)"]) if "intensity (dB)" in s.header else np.nan,
                     desired_rate=float(s.header["desired firing rate (Hz)"]) if "desired firing rate (Hz)" in s.header else np.nan,
                     pause_ms=float(s.header["pause between stimuli (msec)"]) if "pause between stimuli (msec)" in s.header else np.nan,
                     header_stim=s.header.get("stimulus file", "").split("/")[-1].removesuffix(".dat"),
                     offline=s.offline, n_trials=s.n_trials, rate=r.mean(), rate_sd=r.std(ddof=1),
                     rate_full=ia1.trial_rates(s.trials, 0, 1).mean(), rate_sd_full=ia1.trial_rates(s.trials, 0, 1).std(ddof=1)))
D = pd.DataFrame(rows)
print("Data1 experiments:", D.exp.nunique(), " sessions:", len(D), " offline-sorted sessions (no OEL header):", D.offline.sum())
print("experiments not listed in singlecells.txt:", sorted(set(D.exp) - set(cell_of)))
print("cells (rows of singlecells.txt, plus experiments not listed):", len(cells) + len(set(D.exp) - set(cell_of)),
      " recording dates:", D.date.nunique())
print("\nSessions by stimulus file (cell.dat):")
print(D.groupby("stimulus").agg(sessions=("session", "size"), protocol=("protocol", "first")).to_string())
hdr = D[D.header_stim != ""]
print("\nHeader stimulus agrees with cell.dat in", (hdr.header_stim.str.replace("gauss_", "") == hdr.stimulus.str.replace("gauss_", "")).sum(),
      "of", len(hdr), "sessions with a stimulus line in the header")
d = (D.header_intensity - D.intensity).dropna()
print("header intensity minus cell.dat intensity (dB): n =", len(d), " median |diff|", round(d.abs().median(), 2),
      " max |diff|", round(d.abs().max(), 2), " sessions over 1 dB:", list(D.session[(D.header_intensity - D.intensity).abs() > 1]))
print("desired firing rate in headers:", D.desired_rate.value_counts(dropna=False).to_dict())
print("pause between stimuli (ms) in headers:", D.pause_ms.value_counts(dropna=False).to_dict())
allsp = np.concatenate([np.concatenate(s.trials) for s in sessions])
print("\nspike times (s): min", allsp.min(), " max", allsp.max(), " fraction after 1 s (in the pause):", round((allsp > 1).mean(), 4))
print("trials per session: min", D.n_trials.min(), " max", D.n_trials.max())
print("rate over 200-1000 ms (Hz): min", round(D.rate.min(), 1), " max", round(D.rate.max(), 1))
unsorted = sum(int((np.diff(t) <= 0).any()) for s in sessions for t in s.trials)
print("trials with unsorted or repeated spike times:", unsorted)
isi = np.concatenate([np.diff(t) for s in sessions for t in s.trials])
print("shortest ISI (ms):", round(isi.min() * 1e3, 2), " 0.1 ms quantile of ISIs (ms):", round(np.quantile(isi, 0.001) * 1e3, 2))
for name, sh in shapes.items():
    print(f"stimulus {name}: {len(sh)} samples in the first 1.05 s at 20 kHz, dB re its 10 s maximum, range {sh.min():.1f} to {sh.max():.1f}, SD over 0-1 s {sh[:20000].std():.2f} dB")
# Duplicate recordings: identical spike trains or identical first trials.
seen = collections.defaultdict(list)
for s in sessions:
    seen[np.concatenate(s.trials).round(6).tobytes()].append(s.name)
    seen[("first", s.trials[0].round(6).tobytes())].append(s.name)
print("sessions with identical spike times or identical first trials:", [v for v in seen.values() if len(v) > 1])
# Trial blocks: spikes after the 1 s stimulus fall in the pause. A block holding two trials (a
# missing header) would restart its spike times near 0, so its times would not be sorted (counted
# above), and would hold about twice the spike count of the session's other blocks.
rows = []
for s in sessions:
    n01 = np.array([np.sum(t < 1) for t in s.trials])
    mx = max(t.max() for t in s.trials if len(t))
    rows.append(dict(session=s.name, offline=s.offline, max_t=mx, late_blocks=sum(len(t) > 0 and t.max() > 1.7 for t in s.trials),
                     rate_0_1=n01.mean(), rate_pause=np.mean([np.sum((t >= 1) & (t < 1.7)) for t in s.trials]) / 0.7,
                     rate_after_1_7=np.mean([np.sum(t >= 1.7) for t in s.trials]) / max(1e-9, mx - 1.7),
                     doubled=int((n01 > 1.5 * np.median(n01)).sum())))
blk = pd.DataFrame(rows)
lb = blk[blk.max_t > 1.2]
print("\nsessions with spikes later than 1.2 s:", len(lb), "(off-line sorted:", int(lb.offline.sum()), "of", int(blk.offline.sum()), "); with blocks reaching past 1.7 s:",
      int((lb.late_blocks > 0).sum()), "; latest spike", round(blk.max_t.max(), 3), "s; latest spike in on-line sessions", round(blk.max_t[~blk.offline].max(), 3), "s")
print("  in those sessions, rate 0-1 s median", round(lb.rate_0_1.median(), 1), "Hz; 1-1.7 s median", round(lb.rate_pause.median(), 1),
      "Hz (max", round(lb.rate_pause.max(), 1), "); after 1.7 s median", round(lb.rate_after_1_7.median(), 1), "Hz (max", round(lb.rate_after_1_7.max(), 1), ")")
print("  blocks with more than 1.5 times the session's median spike count in 0-1 s:", int(blk.doubled.sum()), "in", list(blk.session[blk.doubled > 0]),
      "; none over 2 times:", all(max(np.sum(t < 1) for t in s.trials) < 2 * np.median([np.sum(t < 1) for t in s.trials]) for s in sessions))
# Data1/firing_rates: one record per spike file, its number of trials and a rate with a +/- value.
frt = z.read(ia1.ROOT + "firing_rates").decode("latin1")
FR = pd.DataFrame([dict(session=f"{e}_sig{g}", fr_stim=st_, fr_n=int(n), fr_rate=float(r), fr_pm=float(pm)) for e, g, st_, n, r, pm in
                   re.findall(r"\./\S+/(\S+)_sig(\d)_spikes\.dat\s*\n(\S+)\s*\n\(n=(\d+)\) firing rate = ([\d.]+) \+/- ([\d.]+)", frt)])
FRM = FR.merge(D[["session", "n_trials", "rate_full", "rate_sd", "offline", "protocol"]], on="session")
print("\nfiring_rates: records", len(FR), "of", len(D), "spike files; trials equal to the blocks parsed here in", int((FRM.fr_n == FRM.n_trials).sum()),
      "; differing:", FRM.loc[FRM.fr_n != FRM.n_trials, ["session", "fr_n", "n_trials"]].values.tolist())
print("  rate / rate over 0-1 s: median", round((FRM.fr_rate / FRM.rate_full).median(), 3), "(on-line sessions", round((FRM.fr_rate / FRM.rate_full)[~FRM.offline].median(), 3),
      "; off-line", round((FRM.fr_rate / FRM.rate_full)[FRM.offline].median(), 3), ")")
print("  +/- value: median", round(FR.fr_pm.median(), 1), "Hz, range", FR.fr_pm.min(), "to", FR.fr_pm.max(), "; trial-to-trial SD of the rate here: median",
      round(D.rate_sd.median(), 1), "Hz; seven-protocol sessions without a record:", sorted(set(D.session[D.protocol.notna()]) - set(FR.session)))

**Trial blocks.** Spikes reach 2.76 s, later than one trial and its pause, so a block without a
header could hold two trials. The check above shows that late spikes occur only in off-line
sorted sessions, at the low rate of the pause, and that no block restarts its times or holds
twice the spike count of the others: each block is one trial and the trial counts stand (the data
set's own `firing_rates` file gives the same counts in 136 of 138 files). The `firing_rates` file
is not a selection: its +/- value is far above any trial-to-trial SD of the rate.

The stimulus files are named for their SD and cutoff (`gauss_st6_co200` is SD 6 dB, cutoff 200
Hz). Their SDs over the first second can be checked against the names (6 dB files near 6 dB).

**Clock check.** Spikes and stimuli share the trial's clock if the spike-triggered average of the
stimulus (STA) shows the upstroke a few ms before the spike, which the paper describes (Figure 1:
"responses typically appear after stimulus upstrokes with an delay of 3.4 msec"). The contrast is
the same STA computed from the stimulus shifted by 100 ms, which keeps every property of the
stimulus and the spike train except their alignment.

In [ ]:
P = [s for s in sessions if s.protocol]          # sessions in the paper's seven protocols


def stim_db(s):
    return shapes[s.stimulus] + s.intensity


def spikes_in(s, t0=ia1.T0, t1=ia1.T1):
    return np.concatenate([t[(t >= t0) & (t < t1)] for t in s.trials])


rows = []
for s in P:
    st = stim_db(s)
    sp = spikes_in(s)
    seg, tt = ia1.segments(st, sp)
    sta = seg.mean(0) - st[4000:20000].mean()
    seg2, _ = ia1.segments(st, sp - 0.1)
    sta2 = seg2.mean(0) - st[4000:20000].mean()
    rows.append(dict(session=s.name, protocol=s.protocol, sta_peak=sta.max(), lat_ms=-tt[np.argmax(sta[tt <= 0])] * 1e3,
                     shifted_peak=sta2.max()))
C = pd.DataFrame(rows)
print("STA peak above the mean stimulus (dB): median", round(C.sta_peak.median(), 2), " range", round(C.sta_peak.min(), 2), "to", round(C.sta_peak.max(), 2))
print("with the stimulus shifted by 100 ms: median", round(C.shifted_peak.median(), 2), " range", round(C.shifted_peak.min(), 2), "to", round(C.shifted_peak.max(), 2))
print("sessions whose STA peak exceeds the shifted one:", (C.sta_peak > C.shifted_peak).sum(), "of", len(C))
print("latency of the STA maximum (ms): median", C.lat_ms.median(), " range", C.lat_ms.min(), "to", C.lat_ms.max())
print("sessions where the true STA peak is under 1 dB:", list(C.session[C.sta_peak < 1]))

## Session selection and burst identification

Kept sessions (claim data-sessions-and-trials). The paper keeps sessions whose trial-to-trial SD
of the firing rate is below 35 Hz and whose mean rate is about 100 Hz. Readings, from the
criterion: (a) the SD condition alone; (b) the SD condition and a target rate of 100 Hz where
the header records one. Per-trial rates over 200 to 1000 ms, and over the whole 1 s as a
variant. Only sessions of the paper's seven protocols (Table 1) can be kept; the data set's other
stimuli (`gaussModality_co200`, `lmd_co200`, cutoffs 50 and 1600 Hz) are not in the paper's design
(Methods: "The other signal could be one of six different stimulation protocols").

Readings (a) and (b) keep 115 and 113 of the 116 seven-protocol sessions, against the paper's
86, so the paper removed sessions by a rule it does not state. Reading (c), added after the first
verification, is the SD condition and two exclusions the data suggest: sessions with fewer than
98 trials (the Methods give "N ranging between 98 and 503" for the paper's sessions, so a session
with fewer was not among them) and sessions whose spikes were detected off-line (`OFFOEL`
header; the data descriptor says off-line detection was done where the on-line detection "was
deemed to be inexact"). The off-line exclusion has no passage in the paper behind it; it is taken
because, with it, the kept sessions per protocol and the bursting sessions per protocol come
close to Table 1 (below). Reading (c) is the main population from here on, an assumption; the
results over the 115 sessions of reading (a) are shown beside it.

Bursts, by the Methods (Burst identification), under every combination of the readings of the
open details (claims file, opening comment): peak (a) first local maximum, (b) largest value in
0 < tau < 5 ms, and (c) the first local maximum after C_s first rises significantly above its
refractory plateau (added after the first verification, see `ia1.py`); minimum (a) first
significant local minimum after the peak, (b) lowest value up to 1.25 / cutoff; error bar N (a)
the number of product terms, (b) the number of trials, and (c) sigma the SD over trials of each
trial's C_s with N the number of trials (added here, see `ia1.py`).

In [ ]:
def kept(reading="a", window="200-1000"):
    col = "rate_sd" if window == "200-1000" else "rate_sd_full"
    out = []
    for s in P:
        row = D.loc[D.session == s.name].iloc[0]
        ok = row[col] < 35
        if reading == "b" and not np.isnan(row.desired_rate):
            ok = ok and row.desired_rate == 100
        if reading == "c":
            ok = ok and not s.offline and s.n_trials >= 98
        if ok:
            out.append(s)
    return out


T1_SESS = [43, 6, 7, 7, 8, 7, 8]
sv = D[D.protocol.notna() & (D.rate_sd < 35)]
rows = []
for label, g in [("SD < 35 Hz (reading a)", sv), ("  without off-line sorted sessions", sv[~sv.offline]),
                 ("  without sessions under 98 trials", sv[sv.n_trials >= 98]),
                 ("  without both (reading c)", sv[~sv.offline & (sv.n_trials >= 98)]),
                 ("  without the 450 ms pause sessions", sv[sv.pause_ms != 450]),
                 ("  without off-line and 450 ms pause", sv[~sv.offline & (sv.pause_ms != 450)]),
                 ("  without off-line, 450 ms pause and under 98 trials", sv[~sv.offline & (sv.pause_ms != 450) & (sv.n_trials >= 98)])]:
    v = [int((g.protocol == p).sum()) for p in range(1, 8)]
    rows.append(dict(selection=label, sessions=len(g), **{f"p{p}": x for p, x in zip(range(1, 8), v)},
                     within_10pct=sum(abs(x - e) <= 0.1 * e for x, e in zip(v, T1_SESS)),
                     min_trials=g.n_trials.min(), mean_trials=round(g.n_trials.mean(), 1)))
rows.append(dict(selection="paper (Table 1)", sessions=86, **{f"p{p}": x for p, x in zip(range(1, 8), T1_SESS)}))
print(pd.DataFrame(rows).to_string(index=False))
print("off-line sorted seven-protocol sessions by protocol:", sv[sv.offline].protocol.value_counts().sort_index().to_dict(),
      "; sessions under 98 trials:", sv.loc[sv.n_trials < 98, ["session", "n_trials", "offline"]].values.tolist())

KEPT = kept("c")
KEPT_A = kept("a")
POP = {"c": {s.name for s in KEPT}, "a": {s.name for s in KEPT_A}}
COMBOS = list(itertools.product("abc", "ab", "abc"))
MAIN = ("b", "a", "a")
limits = {c: {s.name: ia1.burst_limit(s.trials, s.cutoff, *c)[0] for s in P} for c in COMBOS}
for c in COMBOS:
    lim = limits[c]
    nb = sum(lim[s.name] > 0 for s in KEPT)
    na = sum(lim[s.name] > 0 for s in KEPT_A)
    print("peak %s, minimum %s, N %s: bursting %3d of %d kept (c), %3d of %d (a); median limiting ISI %s ms" % (
        *c, nb, len(KEPT), na, len(KEPT_A), np.round(np.median([lim[s.name] for s in KEPT if lim[s.name] > 0]) * 1e3, 2) if nb else "-"))
LIM = limits[MAIN]
BURSTING = [s for s in KEPT if LIM[s.name] > 0]
BURSTING_A = [s for s in KEPT_A if LIM[s.name] > 0]
print("\nmain reading", MAIN, ": kept", len(KEPT), " bursting", len(BURSTING), "; reading (a) of the selection: kept", len(KEPT_A), " bursting", len(BURSTING_A))

## data-sessions-and-trials

> Thirty-seven receptor cells were recorded from 23 animals, giving 132 sessions, of which 86 met
> the firing-rate and stationarity conditions; each stimulus was repeated 98 to 503 times, 172 on
> average.

Data1 does not record the animal. The recording date is shown as a stand-in (one animal per
day is an assumption the descriptor does not make); cells are the rows of `singlecells.txt`.

In [ ]:
def describe(sel, label):
    names = {s.name for s in sel}
    d = D[D.session.isin(names)]
    # cells: rows of singlecells.txt, plus each experiment it does not list
    return dict(population=label, cells=d.cell[d.cell >= 0].nunique() + d.exp[d.cell < 0].nunique(),
                dates=d.date.nunique(), sessions=len(d), min_trials=d.n_trials.min(), max_trials=d.n_trials.max(),
                mean_trials=round(d.n_trials.mean(), 1), mean_rate_sd=round(d.rate_sd.mean(), 2),
                mean_rate_sd_full=round(d.rate_sd_full.mean(), 2))


tab = pd.DataFrame([describe(sessions, "all Data1 sessions"),
                    describe([s for s in sessions if s.stimulus.replace("gauss_", "").startswith("st")], "Gaussian AM stimuli (any cutoff)"),
                    describe(P, "seven protocols"),
                    describe(kept("a"), "kept (a), 200-1000 ms"), describe(kept("b"), "kept (b), 200-1000 ms"),
                    describe(kept("a", "full"), "kept (a), 0-1000 ms"), describe(kept("b", "full"), "kept (b), 0-1000 ms"),
                    describe(kept("c"), "kept (c), 200-1000 ms"), describe(kept("c", "full"), "kept (c), 0-1000 ms")])
print(tab.to_string(index=False))
print("\nsessions in the seven protocols with rate SD >= 35 Hz (200-1000 ms):",
      list(D.session[D.protocol.notna() & (D.rate_sd >= 35)]))
print("SD of the trial rates over the seven-protocol sessions (Hz): quantiles 0, 0.25, 0.5, 0.75, 0.9, 1:",
      np.quantile(D.rate_sd[D.protocol.notna()], [0, .25, .5, .75, .9, 1]).round(1))
expected = dict(cells=(37, 3.7), sessions_recorded=(132, 13.2), kept=(86, 8.6), min_trials=(98, 9.8), max_trials=(503, 50.3),
                mean_trials=(172, 17.2), mean_rate_sd=(6, 0.6))
print("\nexpected (value, tolerance):", expected)

**Result: partially reproduced.** The 116 sessions of the seven protocols come from 37 cells
recorded on 23 dates, the paper's 37 cells and 23 animals (if one animal was recorded per date, an
assumption, as Data1 does not record the animal). The 122 Gaussian AM sessions (any cutoff) are
within 10% of the paper's 132 sessions; all 150 Data1 sessions are not. Under the criterion's
readings the selection does not reproduce: the SD < 35 Hz condition keeps 115 of the 116 sessions
(113 with the 100 Hz target of reading (b)), against 86, and over them the trials range from 69
(outside 98 +/- 9.8) to 503 and average 160.3 (within 172 +/- 17.2), with a mean trial-to-trial
SD of the rate of 6.48 Hz (within 6 +/- 0.6). Reading (c), which also leaves out the sessions with
fewer than 98 trials and those sorted off-line, keeps 89 sessions with 98 to 503 trials (mean
168.7) and a mean SD of 6.00 Hz, all within tolerance. It adds two exclusions the paper does not
give as conditions, chosen in part from these counts (its fewest trials are 98 by construction),
so it does not decide this claim; it is used as the population of the claims that follow.

## table1-sessions-per-protocol

> The 86 kept sessions divide among the seven stimulus protocols as 43 (6 dB, 200 Hz), 6 (3 dB,
> 200 Hz), 7 (12 dB, 200 Hz), 7 (6 dB, 25 Hz), 8 (6 dB, 100 Hz), 7 (6 dB, 400 Hz) and 8 (6 dB, 800 Hz).

In [ ]:
rows = []
for label, sel in [("kept (a) 200-1000", kept("a")), ("kept (b) 200-1000", kept("b")),
                   ("kept (a) 0-1000", kept("a", "full")), ("kept (b) 0-1000", kept("b", "full")),
                   ("kept (c) 200-1000", kept("c")), ("kept (c) 0-1000", kept("c", "full")), ("seven protocols", P)]:
    cnt = collections.Counter(s.protocol for s in sel)
    vals = [cnt[p] for p in range(1, 8)]
    rows.append(dict(population=label, **{f"p{p}": v for p, v in zip(range(1, 8), vals)},
                     within_10pct=sum(abs(v - e) <= 0.1 * e for v, e in zip(vals, T1_SESS))))
rows.append(dict(population="paper", **{f"p{p}": v for p, v in zip(range(1, 8), T1_SESS)}, within_10pct=7))
print(pd.DataFrame(rows).to_string(index=False))
exps = collections.defaultdict(list)
for s in P:
    exps[s.exp].append(s.protocol)
print("\nexperiments with both stimuli in the protocols, by pair:",
      dict(collections.Counter(tuple(sorted(v)) for v in exps.values() if len(v) == 2)))

**Result: not reproduced.** None of the seven counts is within 10% of Table 1 under the
criterion's readings: the kept sessions are 60, 9, 8, 10, 11, 8 and 9 per protocol (reading (a),
200 to 1000 ms) against 43, 6, 7, 7, 8, 7 and 8. Under reading (c) of the selection they are 47,
6, 7, 7, 8, 7 and 7, six of seven within tolerance (800 Hz has 7 against 8), but that reading was
chosen in part from these counts. The data set holds every reference session paired with each
other protocol (55 experiments) and more reference sessions than the paper counts.

## firing-rate-uniform

> The kept sessions had a mean firing rate of 113 Hz (SD 16 Hz) that did not differ significantly
> across stimulus conditions (one-way ANOVA, p = 0.58).

In [ ]:
rows = []
for label, sel, col in [("kept (c), 200-1000 ms", kept("c"), "rate"), ("kept (c), 0-1000 ms", kept("c", "full"), "rate_full"),
                        ("kept (a), 200-1000 ms", kept("a"), "rate"), ("kept (a), 0-1000 ms", kept("a", "full"), "rate_full"),
                        ("all seven-protocol sessions, 200-1000 ms", P, "rate"), ("all seven-protocol sessions, 0-1000 ms", P, "rate_full"),
                        ("kept (b), 200-1000 ms", kept("b"), "rate")]:
    d = D[D.session.isin({s.name for s in sel})]
    groups = [g[col].values for _, g in d.groupby("protocol")]
    F, p = stats.f_oneway(*groups)
    rows.append(dict(population=label, n=len(d), mean_rate=round(d[col].mean(), 1), sd_rate=round(d[col].std(ddof=1), 1),
                     F=round(F, 2), p=round(p, 3)))
print(pd.DataFrame(rows).to_string(index=False))
d = D[D.session.isin({s.name for s in KEPT})]
print("\nmean rate by protocol (kept (c), 200-1000 ms):", d.groupby("protocol").rate.mean().round(1).to_dict())

**Result: reproduced.** Over the 89 sessions of selection (c) the mean rate is 107.1 Hz (within
113 +/- 16) with SD 14.5 Hz (within 16 +/- 1.6), and a one-way ANOVA across the seven protocols
finds no difference (F = 0.49, p = 0.814). Over the 115 sessions of selection (a) the mean is
110.2 Hz (SD 16.7), F = 0.49, p = 0.818; over all seven-protocol sessions and with whole-trial
rates the ANOVA gives p between 0.471 and 0.809. No population shows a significant difference.

## threshold-population (blocked)

> The mean threshold of the cells at their best frequency was 58 dB (SD 14 dB).

Data1 holds spike times, `cell.dat` (stimulus names and the intensities they were played at) and
the stimuli. It holds no threshold, best-frequency or tuning (f-I) measurement: the listing below
shows every kind of file in Data1. Blocked, `data_missing`.

In [ ]:
kinds = collections.Counter()
for n in z.namelist():
    if n.startswith(ia1.ROOT) and not n.endswith("/"):
        base = n.split("/")[-1]
        kinds[base.split("_")[-1] if "_" in base and n.count("/") > 3 and "stimuli" not in n else ("stimuli/" + base if "stimuli" in n else base)] += 1
print(dict(kinds))
print("cell.dat keys:", sorted({line.split(":")[0].strip() for n in z.namelist() if n.endswith("cell.dat")
                               for line in z.read(n).decode("latin1").splitlines() if ":" in line}))

## table1-bursting-sessions

> The number of sessions with bursts of n > 1 depends on the stimulus, from none for 3 dB and for
> 800 Hz to all 7 for 12 dB and 7 of 8 for 100 Hz; 40 of 43 reference sessions, 6 of 7 at 25 Hz
> and 3 of 7 at 400 Hz.

Counted over the kept sessions, under every combination of the burst-identification readings. The
fraction of sessions is shown beside the counts, since the counts depend on how many sessions are
kept.

In [ ]:
T1_BURST = [40, 0, 7, 6, 7, 3, 0]
tol = [max(0.1 * e, 0) for e in T1_BURST]


def n_gt1_sessions(sel, lim):
    cnt = collections.Counter()
    for s in sel:
        if lim[s.name] > 0 and (ia1.burst_counts(s.trials, lim[s.name])[1:] > 0).any():
            cnt[s.protocol] += 1
    return [cnt[p] for p in range(1, 8)]


rows = []
for kr in ["c", "a", "b"]:
    sel = kept(kr)
    tot = collections.Counter(s.protocol for s in sel)
    for c in COMBOS:
        v = n_gt1_sessions(sel, limits[c])
        rows.append(dict(kept=kr, readings="".join(c), **{f"p{p}": f"{x}/{tot[p]}" for p, x in zip(range(1, 8), v)},
                         within=sum(abs(x - e) <= t for x, e, t in zip(v, T1_BURST, tol))))
print(pd.DataFrame(rows).to_string(index=False))
print("paper:", [f"{a}/{b}" for a, b in zip(T1_BURST, T1_SESS)])

**Result: partially reproduced** (resting on selection (c) and the burst readings). Under the main
burst reading (peak b, minimum a, N a) the sessions of selection (c) with bursts of n > 1 are
40/47, 1/6, 7/7, 6/7, 7/8, 3/7 and 0/7 against 40/43, 0/6, 7/7, 6/7, 7/8, 3/7 and 0/8: six of the
seven counts equal the paper's, and one 3 dB session bursts where the paper has none. The first
local maximum read with the refractory plateau left out (peak c) gives 37/47 reference sessions
and the same other counts (six of seven within tolerance). Over the 115 sessions of selection (a)
at most 2 of 7 are within tolerance (52/60, 2/9, 8/8, 9/10, 10/11, 3/8, 0/9 under the main
reading). Under peak reading (a) 5 of 47 reference sessions burst, and with N the number of trials
(b) none does.

## table1-isolated-spikes

> The percentage of isolated spikes among all bursts of the bursting sessions depends on the
> stimulus (74, 100, 62, 60, 55, 96 and 100% for protocols 1 to 7), and in every protocol isolated
> spikes are the most probable burst.

In [ ]:
T1_ISO = [74, 100, 62, 60, 55, 96, 100]


def protocol_counts(sel, lim):
    # Pooled n counts per protocol over the bursting sessions (all sessions where a protocol has none).
    out = {}
    for p in range(1, 8):
        ses = [s for s in sel if s.protocol == p]
        bur = [s for s in ses if lim[s.name] > 0]
        use = bur if bur else ses
        tot = np.zeros(400)
        per = []
        for s in use:
            c = ia1.burst_counts(s.trials, lim[s.name])
            tot[:len(c)] += c
            per.append(c[0] / c.sum())
        out[p] = (np.trim_zeros(tot, "b"), per, len(bur))
    return out


rows = []
PC = {}
for kr, sel, c in [("c", KEPT, c) for c in COMBOS] + [("a", KEPT_A, c) for c in COMBOS]:
    pc = protocol_counts(sel, limits[c])
    PC[(kr, c)] = pc
    iso = [100 * pc[p][0][0] / pc[p][0].sum() for p in range(1, 8)]
    iso_avg = [100 * np.mean(pc[p][1]) for p in range(1, 8)]
    most = [int(np.argmax(pc[p][0]) + 1) for p in range(1, 8)]
    rows.append(dict(kept=kr, readings="".join(c), **{f"p{p}": round(v, 1) for p, v in zip(range(1, 8), iso)},
                     within=sum(abs(v - e) <= 0.1 * e for v, e in zip(iso, T1_ISO)), most_probable_n=most,
                     session_average=[round(v, 1) for v in iso_avg]))
print(pd.DataFrame(rows).to_string(index=False))
print("paper:", T1_ISO)

**Result: not reproduced.** Under the main reading the pooled percentage of isolated spikes over
selection (c) is 80.7, 71.7, 66.1, 78.0, 71.3, 95.0 and 100% against 74, 100, 62, 60, 55, 96 and
100%: 4 of 7 within tolerance (protocols 1, 3, 6, 7). At 3 dB the one bursting session gives 71.7%
where the paper has no bursts, and 25 Hz and 100 Hz have more isolated spikes than the paper. No
combination of readings brings more than 5 of 7 within tolerance (4 over selection (a)). n = 1 is
the most probable burst in every protocol under the main reading, as the paper says. The criterion
requires every value; 4 of 7 is below the five of seven at which the other Table 1 criteria call a
result partial, so the outcome is not reproduced (the first round recorded it as partially
reproduced).

## table1-burst-length

> The highest n and the average n depend on the stimulus; the longest bursts (up to 15 spikes,
> average 2.10 with SD 3.67) occur at 25 Hz, and no bursts occur at 3 dB or at 800 Hz.

In [ ]:
EXP_HI = [9, 1, 8, 15, 9, 3, 1]
EXP_AV = [1.3, 1, 1.5, 2.1, 1.7, 1.04, 1]
EXP_SD = {1: 1.3, 3: 1.6, 4: 3.67, 5: 1.8, 6: 0.42}
rows = []
for kr, c in PC:
    pc = PC[(kr, c)]
    hi = [len(pc[p][0]) for p in range(1, 8)]
    av = [ia1.average_n(pc[p][0])[0] for p in range(1, 8)]
    sd = [ia1.average_n(pc[p][0])[1] for p in range(1, 8)]
    ok = (sum(abs(h - e) <= 0.1 * e for h, e in zip(hi, EXP_HI)) + sum(abs(a - e) <= 0.1 * e for a, e in zip(av, EXP_AV))
          + sum(abs(sd[p - 1] - e) <= 0.1 * e for p, e in EXP_SD.items()))
    rows.append(dict(kept=kr, readings="".join(c), highest_n=hi, average_n=[round(a, 2) for a in av], sd_n=[round(x, 2) for x in sd], within_of_19=ok))
print(pd.DataFrame(rows).to_string(index=False))
print("paper: highest", EXP_HI, " average", EXP_AV, " SD", EXP_SD)

**Result: not reproduced.** Under the main reading over selection (c) 8 of the 19 values are
within tolerance (highest n 7, 9, 8, 14, 7, 5, 1 against 9, 1, 8, 15, 9, 3, 1; average n 1.23,
1.39, 1.49, 1.90, 1.39, 1.05, 1.00 against 1.3, 1, 1.5, 2.1, 1.7, 1.04, 1), below the 10 the
criterion needs for a partial reproduction, and no combination of readings under either selection
gives more than 9 (peak reading c). The longest bursts are at 25 Hz (14 spikes) and 800 Hz has
none, as in the paper, but 3 dB has bursts of up to 9 spikes in its one bursting session.

## fig5-burst-probability-vs-stimulus

> The probability of generating isolated spikes is lowest, and high-n bursts most frequent, for
> large amplitude fluctuations (12 dB) and cutoff frequencies around 100 Hz.

In [ ]:
def pn_table(sel, lim, bursting_only):
    rows = []
    for s in sel:
        if bursting_only and lim[s.name] <= 0:
            continue
        c = ia1.burst_counts(s.trials, lim[s.name]).astype(float)
        c = np.r_[c, np.zeros(3)][:3] / c.sum()
        rows.append(dict(protocol=s.protocol, P1=c[0], P2=c[1], P3=c[2]))
    return pd.DataFrame(rows).groupby("protocol").agg(["mean", "std", "size"])


for kr, sel, c in [("c", KEPT, MAIN), ("c", KEPT, ("b", "b", "a")), ("c", KEPT, ("b", "a", "c")), ("c", KEPT, ("a", "a", "a")),
                  ("c", KEPT, ("c", "a", "a")), ("a", KEPT_A, MAIN)]:
    for reading, bo in [("(a) all kept", False), ("(b) bursting only", True)]:
        t = pn_table(sel, limits[c], bo)
        means = t.xs("mean", axis=1, level=1)
        cut = {25: 4, 100: 5, 200: 1, 400: 6, 800: 7}
        sdp = {3: 2, 6: 1, 12: 3}
        p1c = {k: means.P1.get(v, np.nan) for k, v in cut.items()}
        p1s = {k: means.P1.get(v, np.nan) for k, v in sdp.items()}
        print(f"kept ({kr}), readings {''.join(c)}, sessions {reading}")
        print("  P(n=1) by cutoff:", {k: round(v, 3) for k, v in p1c.items()}, " lowest at", min(p1c, key=p1c.get), "Hz")
        print("  P(n=1) by SD:", {k: round(v, 3) for k, v in p1s.items()}, " lowest at", min(p1s, key=p1s.get), "dB")
        for col in ["P2", "P3"]:
            print(f"  {col} by cutoff:", {k: round(means[col].get(v, np.nan), 3) for k, v in cut.items()},
                  " by SD:", {k: round(means[col].get(v, np.nan), 3) for k, v in sdp.items()})
t = pn_table(KEPT, LIM, False)
print("\nmain reading, all kept (c): mean, SD and number of sessions per protocol")
print(t.round(3).to_string())

**Result: reproduced.** Under the main reading over selection (c) the population average of
P(n = 1) is lowest at 100 Hz among the cutoffs (0.749, against 0.816 at 25, 0.837 at 200, 0.980
at 400 and 1.000 at 800 Hz) and lowest at 12 dB among the SDs (0.668, against 0.837 at 6 and 0.953
at 3 dB), with all kept sessions and with bursting sessions only; the same holds over selection
(a) (0.755 and 0.686) and with the plateau-excluded peak (c). P(n = 2) and P(n = 3) are highest at
100 Hz and at 12 dB over all kept sessions. With the bounding minimum read as the lowest value
(b) the lowest P(n = 1) moves to 25 Hz, where that reading gives very long bursts.

## fig3-correlation-shape

> For a cell tested with all stimuli, larger stimulus SD gives correlation functions with sharper
> peaks, and faster stimuli lead from multi-modal (25 Hz) to single-peaked (100, 200 Hz) to
> shallower and broader correlation functions (400, 800 Hz).

`singlecells.txt` lists one cell recorded in seven experiments, 03-04-23-ad to -aj, which together
hold all seven protocols (six reference sessions, one of each other protocol): this is taken as the
cell of Figure 3. C_s is normalised by the squared mean rate so that sessions compare. "Sharper"
is read as (a) a higher first peak over the baseline (the mean of C_s/rbar^2 over 20 to 40 ms) and
(b) a narrower first peak (full width at half height above that baseline). Peaks are counted on
C_s/rbar^2 smoothed with a 1 ms running mean (a standard choice that keeps the ms-scale shape), as
local maxima within 0 < tau <= 1.25 / cutoff that rise above the neighbouring minima by more than a
prominence threshold; the threshold is not in the paper, so 0.05, 0.1 and 0.2 are shown.

In [ ]:
from scipy.signal import find_peaks

fig3_cell = next(row for row in cells if "03-04-23-ad" in row)
F3 = [s for s in P if s.exp in fig3_cell]
rows = []
curves = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for s in F3:
    tau, c, sd, count, ntr, sdt = ia1.correlation(s.trials, 0.06)
    rb = ia1.binned(s.trials).astype(bool).mean() / ia1.DT
    cn = c / rb ** 2
    base = cn[(tau >= 0.02) & (tau <= 0.04)].mean()
    sm = np.convolve(cn, np.ones(10) / 10, mode="same")
    sm[:6] = cn[:6]
    lim_i = int(1.25 / s.cutoff / ia1.DT)
    ip = 1 + np.argmax(cn[1:51])
    height = cn[ip] - base
    width = ia1.fwhm(cn, ip, base) * ia1.DT * 1e3
    npk = {}
    for prom in [0.05, 0.1, 0.2]:
        pk, _ = find_peaks(sm[1:lim_i + 1], prominence=prom)
        npk[prom] = len(pk)
    rows.append(dict(session=s.name, sd=s.sd_cut[0], cutoff=s.cutoff, bursting=LIM[s.name] > 0,
                     limit_ms=round(LIM[s.name] * 1e3, 1), peak_ms=round(tau[ip] * 1e3, 1), peak_height=round(height, 2),
                     peak_fwhm_ms=round(width, 1), **{f"peaks_prom{k}": v for k, v in npk.items()}))
    curves.append(cn[1:])
    ax = axes[0] if s.sd_cut[1] == 200 else axes[1]
    ax.plot(tau[1:] * 1e3, cn[1:], lw=0.8, label=f"{s.sd_cut[0]} dB, {s.cutoff} Hz ({s.exp[-2:]})")
F3T = pd.DataFrame(rows)
print(F3T.to_string(index=False))
for ax, t in zip(axes, ["cutoff 200 Hz", "SD 6 dB"]):
    ax.set(xlabel="tau (ms)", ylabel="C_s / rbar^2", title=t, ylim=(-1.5, 4))
    ax.legend(fontsize=7)
allc = np.concatenate(curves)
print("C_s/rbar^2 for tau > 0: range", allc.min().round(2), "to", allc.max().round(2), "; points outside the axes (-1.5 to 4):", int(((allc < -1.5) | (allc > 4)).sum()))
plt.show()
ref = F3T[(F3T.sd == 6) & (F3T.cutoff == 200)]
print("reference sessions: peak height range", ref.peak_height.min(), "to", ref.peak_height.max(), "; FWHM range", ref.peak_fwhm_ms.min(), "to", ref.peak_fwhm_ms.max(), "ms")

**Result: reproduced** (resting on the peak-counting threshold). In the cell recorded with all
seven protocols (03-04-23-ad to -aj) the first peak of C_s/rbar^2 rises above its baseline by 0.50
at 3 dB, 1.14 to 2.35 in the six 6 dB sessions and 5.06 at 12 dB, so it sharpens with SD under
reading (a); its width (b) does not order (0.4, 0.4 to 0.5, 0.3 ms). At 25 Hz C_s has 4 to 14
peaks, at 200 Hz one; at 100 Hz one peak at prominence 0.2 and two at 0.05 and 0.1. The 3 dB, 400
Hz and 800 Hz sessions have no separable first peak, as the paper names for Figures 3A, E and F.

## method-correlation-error

> The temporal average of the estimated error of the correlation function was on average 3.4% (SD
> 1.5%) of the span of C_s.

Delta = sigma / sqrt(N) at each tau, averaged over tau, over max minus min of C_s, averaged over
kept sessions. Readings: sigma and N (a), (b) and (c); tau range (a) 0 to 1.25 / cutoff, (b) the
whole range up to 800 ms. Whether tau = 0, where C_s holds the self term (rbar / dt, two orders of
magnitude above the rest), enters the span is not stated: both are shown.

In [ ]:
rows = []
for s in KEPT_A:
    tau, c, sd, count, ntr, sdt = ia1.correlation(s.trials, 0.7999)
    L = int(round(1.25 / s.cutoff / ia1.DT))
    for nm in "abc":
        err = ia1.correlation_error(sd, count, ntr, sdt, nm)
        for rng_name, sl in [("(a) 0 to 1.25/cutoff", slice(0, L + 1)), ("(b) 0 to 800 ms", slice(0, None))]:
            for z0 in [True, False]:
                ss = sl if z0 else slice(1, sl.stop)
                rows.append(dict(session=s.name, kept_c=s.name in POP["c"], N=nm, tau_range=rng_name, tau0=z0,
                                 pct=100 * err[ss].mean() / (c[ss].max() - c[ss].min())))
E = pd.DataFrame(rows)
print("kept (c)")
print(E[E.kept_c].groupby(["N", "tau_range", "tau0"]).pct.agg(["mean", "std", "size"]).round(2).to_string())
print("\nkept (a)")
print(E.groupby(["N", "tau_range", "tau0"]).pct.agg(["mean", "std", "size"]).round(2).to_string())

**Result: reproduced** (resting on leaving tau = 0 out of the span). With sigma the SD of the
product terms and N their number (reading a), over the whole range of lags and without the
zero-lag self term, the error averages 2.63% (SD 1.39%) of the span of C_s over the 89 sessions of
selection (c), within 3.4 +/- 1.5%; reading (c) of sigma and N gives 2.61%, and over the 115
sessions of selection (a) 2.69% and 2.67%. With tau = 0 in the span every reading is below 0.2%
or, with N the number of trials, 6.3 to 7.9%; over 0 to 1.25 / cutoff without tau = 0 the values
are unstable (means 30 to 4500%).

## model-matches-classification (blocked)

> A threshold-linear Poisson model with refractoriness and no free parameters classifies sessions
> as bursting or not in agreement with the real data in 86% of sessions, and where both are
> bursting their limiting ISIs differ by less than 1 ms in 81% of cases.

The model thresholds the stimulus at the cell's measured threshold s_TH before filtering (Eq. 2),
and the criterion says the claim is not testable as the paper ran it if the data set does not hold
s_TH. Data1 holds no threshold (see threshold-population). Blocked, `data_missing`; the dependent
model-reproduces-coding-trends is blocked for the same reason.

## fig6-bursts-less-correlated

> Representing the spike train as n-bursts reduces correlations between time bins; the mean
> squared Pearson coefficient over t from 200 to 990 ms and tau from 0 to 10 ms is 2.89 times
> larger (SD 1.49) for spikes than for bursts, on average over the bursting sessions.

Eq. 11 across trials at each (t, tau). Bin width (a) 0.1 ms, (b) 0.4 ms; tau = 0 included or not.
The spike variable is 1 when the bin holds a spike. Bursting sessions under the main burst
reading; detail (4) reading (b) (bursting sessions that have bursts of n > 1) is shown too.

In [ ]:
rows = []
for s in BURSTING_A:
    for bw in [1e-4, 4e-4]:
        xs = (ia1.binned(s.trials, dt=bw) > 0).astype(np.int8)
        xb = ia1.burst_matrix(s.trials, LIM[s.name], dt=bw)
        nl, te = int(round(0.01 / bw)), int(round(0.79 / bw))
        cs, cb = ia1.sq_pearson_by_lag(xs, nl, te), ia1.sq_pearson_by_lag(xb, nl, te)
        has_n2 = (xb > 1).any()
        rows.append(dict(session=s.name, kept_c=s.name in POP["c"], bin_ms=bw * 1e3, n_gt1=has_n2, with_tau0=cs.mean() / cb.mean(),
                         without_tau0=cs[1:].mean() / cb[1:].mean()))
R6 = pd.DataFrame(rows)
summ = []
for (kr, R), bw in itertools.product([("c", R6[R6.kept_c]), ("a", R6)], [0.1, 0.4]):
    for lab, g in [("(a) all bursting", R[R.bin_ms == bw]), ("(b) with n > 1", R[(R.bin_ms == bw) & R.n_gt1])]:
        for col in ["with_tau0", "without_tau0"]:
            summ.append(dict(kept=kr, bin_ms=bw, sessions=lab, tau0=col, n=len(g), mean_ratio=round(g[col].mean(), 2),
                             sd_ratio=round(g[col].std(ddof=1), 2), frac_above_1=round((g[col] > 1).mean(), 3),
                             min=round(g[col].min(), 2), max=round(g[col].max(), 2)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: reproduced.** Over the 64 bursting sessions of selection (c) the mean squared Pearson
coefficient is larger for spikes than for bursts in every session; the mean ratio is 1.99 (SD
0.92) with 0.1 ms bins and 2.17 (SD 1.86) with 0.4 ms bins when tau = 0 is left out, within 2.89
+/- 1.49, and 1.29 and 1.21 when it is included (outside). Over the 84 bursting sessions of
selection (a) the ratios are 1.99 and 2.13 without tau = 0. Every bursting session has bursts of
n > 1, so reading (b) of the sessions is the same.

## Information per burst (Figure 7): per-session measures

For every kept session, under the main burst reading, with the bin widths (a) 0.1 ms and (b) 0.4
ms: rbar_n, I_n(1) in bits per event (reading i; reading ii is T = 0.8 s times it), I'_n in bits/s,
and the jitter of each n (readings (a) and (b) of the events). Fits are over the n values present.
Sessions of reading (a) of the selection are computed and summarised under both selections.

The plug-in estimate of I_n(1) is biased upwards for rare events, and more so the rarer they are,
so part of a rise of I_n(1) with n can come from the fall in occurrence. As a baseline (not in the
paper), the same measures are computed after shifting each trial's burst onsets by a random
offset, wrapped within 200 to 1000 ms (`ia1.circular_shift`), which keeps the bursts of each
trial and removes their locking to the stimulus; ten shifts per session, averaged.

In [ ]:
T = ia1.T1 - ia1.T0
rng7 = np.random.default_rng(2008)
info, info_sh = {}, {}
for s in KEPT_A:
    tr, on, n = ia1.bursts(s.trials, LIM[s.name])
    shifted = [ia1.circular_shift(tr, on, s.n_trials, rng7) for _ in range(10)]
    for bw in [1e-4, 4e-4]:
        info[(s.name, bw)] = ia1.information(ia1.rates_of(on, n, s.n_trials, bw))
        sh = [ia1.information(ia1.rates_of(o, n, s.n_trials, bw)) for o in shifted]
        info_sh[(s.name, bw)] = {m: {k: np.mean([x[m][k] for x in sh]) for k in sh[0][m]} for m in sh[0]}


def fit_rows(sel, bw, src):
    rows = []
    for s in sel:
        inf = src[(s.name, bw)]
        ns = np.array(sorted(inf))
        i1 = np.array([inf[n]["i1"] for n in ns])
        rbar = np.array([inf[n]["rbar"] for n in ns])
        rate = np.array([inf[n]["rate"] for n in ns])
        occ = rbar / rbar.sum()
        row = dict(session=s.name, kept_c=s.name in POP["c"], protocol=s.protocol, n_max=ns.max(), n_values=len(ns))
        for name, y in [("i1", i1), ("i1_ii", i1 * T), ("occ_log2", np.log2(occ)), ("occ_ln", np.log(occ)),
                        ("occ_log10", np.log10(occ)), ("rate", rate)]:
            sl, se, pg, p2 = ia1.line_fit(ns, y)
            row.update({f"{name}_slope": sl, f"{name}_se": se, f"{name}_pg": pg, f"{name}_p2": p2})
        row["total_rate"] = rate.sum()
        row["frac_gt1"] = rate[ns > 1].sum() / rate.sum()
        rows.append(row)
    return pd.DataFrame(rows)


F7 = {bw: fit_rows(BURSTING_A, bw, info) for bw in [1e-4, 4e-4]}
F7S = {bw: fit_rows(BURSTING_A, bw, info_sh) for bw in [1e-4, 4e-4]}
for kr in "ca":
    f = F7[1e-4] if kr == "a" else F7[1e-4][F7[1e-4].kept_c]
    print(f"kept ({kr}): bursting sessions {len(f)}; with at least 3 n values (a slope with an error):",
          int((f.n_values >= 3).sum()), "; with 2:", int((f.n_values == 2).sum()), "; with 1:", int((f.n_values == 1).sum()))

## fig7ab-information-per-burst-increases

> The information carried by each n-burst increases with n; every bursting session has a
> significantly positive slope of I_n(1) against n, with an average slope of 1.5 bits (SD 0.7
> bits) over 59 bursting cells.

Significance (a) slope more than one standard error above zero, (b) one-sided t-test p < 0.05.
Sessions with fewer than three n values have no error for the slope and count as not significant.

In [ ]:
rows = []
for kr, bw in itertools.product("ca", [1e-4, 4e-4]):
    f, fs = F7[bw], F7S[bw]
    if kr == "c":
        f, fs = f[f.kept_c], fs[fs.kept_c]
    for norm, col in [("(i) bits per event", "i1"), ("(ii) Eq. 10 as printed", "i1_ii")]:
        for subset, g, gs in [("(a) all bursting", f, fs), ("(b) n_max > 1", f[f.n_max > 1], fs[f.n_max.values > 1])]:
            sig_a = (g[f"{col}_slope"] - g[f"{col}_se"] > 0)
            sig_b = (g[f"{col}_pg"] < 0.05)
            rows.append(dict(kept=kr, bin_ms=bw * 1e3, I1=norm, sessions=subset, n=len(g), mean_slope=round(g[f"{col}_slope"].mean(), 2),
                             sd_slope=round(g[f"{col}_slope"].std(ddof=1), 2), positive=int((g[f"{col}_slope"] > 0).sum()),
                             sig_a=int(sig_a.sum()), sig_b=int(sig_b.sum()), shifted_mean_slope=round(gs[f"{col}_slope"].mean(), 2),
                             slope_minus_shifted=round((g[f"{col}_slope"].values - gs[f"{col}_slope"].values).mean(), 2),
                             above_shifted=int((g[f"{col}_slope"].values > gs[f"{col}_slope"].values).sum())))
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Over the 64 bursting sessions of selection (c), within 59 +/-
5.9, the mean slope of I_n(1) on n is 2.22 bits (SD 0.87) with 0.1 ms bins, just outside 1.5 +/-
0.7, and 1.91 (SD 0.75) with 0.4 ms bins in bits per event, 1.78 and 1.53 for Eq. 10 as printed,
within. All 64 slopes are positive; 63 are significant by one standard error and 62 by the t-test
(one session has only two n values), so not every session. Over the 84 bursting sessions of
selection (a), outside 59 +/- 5.9, the slopes are 2.14, 1.85, 1.71 and 1.48.

The baseline with each trial's burst onsets shifted at random gives a larger mean slope, 2.59 bits
with 0.1 ms bins and 2.23 with 0.4 ms bins (selection (c)); the slope of the data exceeds that of
its shifted copy in only 3 of 64 sessions (9 with 0.4 ms bins). The rise of I_n(1) with n that the
criterion's plug-in formula gives is therefore accounted for by the bias of that estimate for rare
events, not by the locking of high-n bursts to the stimulus. The criterion fixes the formula, so the
outcome rests on it.

## fig7cd-occurrence-decreases

> High-n bursts are rare; in every session the frequency of occurrence decreases significantly with
> n, with an average slope of -2.6 (SD 0.7).

Slope of the logarithm of rbar_n / sum rbar_n on n; logarithm (a) base 2, (b) natural, (c) base
10. Significance as in fig7ab (negative here). The occurrence does not depend on the bin width.

In [ ]:
rows = []
for kr in "ca":
    f = F7[1e-4] if kr == "a" else F7[1e-4][F7[1e-4].kept_c]
    for col in ["occ_log2", "occ_ln", "occ_log10"]:
        for subset, g in [("(a) all bursting", f), ("(b) n_max > 1", f[f.n_max > 1])]:
            rows.append(dict(kept=kr, log=col, sessions=subset, n=len(g), mean_slope=round(g[f"{col}_slope"].mean(), 2),
                             sd_slope=round(g[f"{col}_slope"].std(ddof=1), 2), negative=int((g[f"{col}_slope"] < 0).sum()),
                             sig_a=int((g[f"{col}_slope"] + g[f"{col}_se"] < 0).sum()), sig_b=int(((1 - g[f"{col}_pg"]) < 0.05).sum())))
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Over the 64 bursting sessions of selection (c) the log2
occurrence falls with n with a mean slope of -2.82 (SD 1.12), within -2.6 +/- 0.7 (natural log
-1.96, log10 -0.85, outside); every slope is negative and 63 of 64 are significant by reading (a),
61 by reading (b), so not every session, as the claim states. Over selection (a): -2.71, with 83
and 81 of 84 significant.

## fig7ef-jitter-independent-of-n

> The jitter of the first spike of a burst is roughly independent of n in 80% of the bursting
> sessions, and the mean slope of jitter against n is -0.03 ms (SD 0.2 ms).

Jitter per n is the mean over events of the SD of the onset times. Events (a) peaks of the onset
histogram in 0.1 ms bins, between the troughs on either side; (b) onsets within the session's
first-peak domain (the limiting ISI) centred on an event centre, the centres being the histogram
bins taken from the highest down (`ia1.jitter_events_b`). Slopes over the n values with a jitter (events with at
least two onsets); "not significant" is (a) |slope| below one standard error, (b) two-sided p >=
0.05; sessions with fewer than three such n values have no test and are counted apart.

In [ ]:
rows = []
for s in BURSTING_A:
    tr, on, n = ia1.bursts(s.trials, LIM[s.name])
    row = dict(session=s.name, kept_c=s.name in POP["c"])
    for ev in "ab":
        ns, js = [], []
        for m in np.unique(n):
            obt = [on[(n == m) & (tr == i)] for i in range(s.n_trials)]
            j = ia1.jitter_events_a(obt) if ev == "a" else ia1.jitter_events_b(obt, LIM[s.name])
            if np.isfinite(j):
                ns.append(m)
                js.append(j * 1e3)
        sl, se, pg, p2 = ia1.line_fit(ns, js)
        row.update({f"{ev}_slope": sl, f"{ev}_se": se, f"{ev}_p2": p2, f"{ev}_nvals": len(ns),
                    f"{ev}_j1": js[0] if ns and ns[0] == 1 else np.nan})
    rows.append(row)
J = pd.DataFrame(rows)
summ = []
for kr, ev in itertools.product("ca", "ab"):
    Jk = J[J.kept_c] if kr == "c" else J
    g = Jk[Jk[f"{ev}_nvals"] >= 3]
    summ.append(dict(kept=kr, events=ev, sessions_with_test=len(g), no_test=int((Jk[f"{ev}_nvals"] < 3).sum()),
                     mean_slope_ms=round(Jk[f"{ev}_slope"].mean(), 3), sd_slope_ms=round(Jk[f"{ev}_slope"].std(ddof=1), 3),
                     pct_not_sig_a=round(100 * (g[f"{ev}_slope"].abs() < g[f"{ev}_se"]).mean(), 1),
                     pct_not_sig_b=round(100 * (g[f"{ev}_p2"] >= 0.05).mean(), 1),
                     median_jitter_n1_ms=round(Jk[f"{ev}_j1"].median(), 3)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: reproduced** (resting on how events are formed). Over the bursting sessions of
selection (c), with events read as onsets within the first-peak domain around an event centre (b),
the mean slope of jitter on n is 0.005 ms (SD 0.052), within -0.03 +/- 0.2, and 86.9% of the 61
sessions with a test have a slope not significantly different from 0 by the t-test, inside 80 +/-
8. With events between histogram troughs (a) the slope is -0.001 ms but 88.9% are not significant
(just outside); with significance read as one standard error, 32.8% and 44.4% (outside). Over
selection (a), 87.7% (b) and 91.6% (a). The median jitter of 1-bursts is 0.389 ms under (b) and
0.058 ms under (a), the latter below the paper's 0.2 ms error bar, so reading (a) splits events.

## fig7gh-information-rate

> The information rate of n-bursts decreases with n (average slope -11 bits/s, SD 16 bits/s), yet
> bursts with n > 1 together carry on average 47% of the total information over the bursting cells.

I'_n = rbar_n I_n(1) (bits/s), the same under readings (i) and (ii) of I_n(1).

In [ ]:
rows = []
for kr, bw in itertools.product("ca", [1e-4, 4e-4]):
    f, fs = F7[bw], F7S[bw]
    if kr == "c":
        f, fs = f[f.kept_c], fs[fs.kept_c]
    for subset, g, gs in [("(a) all bursting", f, fs), ("(b) n_max > 1", f[f.n_max > 1], fs[f.n_max.values > 1])]:
        rows.append(dict(kept=kr, bin_ms=bw * 1e3, sessions=subset, n=len(g), mean_slope=round(g.rate_slope.mean(), 1),
                         sd_slope=round(g.rate_slope.std(ddof=1), 1), negative=int((g.rate_slope < 0).sum()),
                         mean_frac_gt1_pct=round(100 * g.frac_gt1.mean(), 1), sd_frac_pct=round(100 * g.frac_gt1.std(ddof=1), 1),
                         mean_total_bits_s=round(g.total_rate.mean(), 1), shifted_slope=round(gs.rate_slope.mean(), 1),
                         shifted_total_bits_s=round(gs.total_rate.mean(), 1), shifted_frac_gt1_pct=round(100 * gs.frac_gt1.mean(), 1)))
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Over the 64 bursting sessions of selection (c) the information rate
falls with n in every session, but the mean slope is -46.5 bits/s (SD 39.0) with 0.1 ms bins and
-38.1 (SD 33.1) with 0.4 ms bins, outside -11 +/- 16, and bursts with n > 1 carry 35.5% and 37.7%
of the information, outside 47 +/- 4.7. The total rate averages 241.4 and 199.8 bits/s. With the
burst onsets shifted at random in each trial the same formula still gives 106.5 and 37.5 bits/s,
slopes of -16.1 and -3.9 bits/s and fractions of 46.8% and 63.2%: a large part of these rates is
estimator bias. Over selection (a): -44.7 and -36.6 bits/s, 36.3% and 38.6%.

## direct-method-comparison

> Information rates from the burst code are highly correlated with those of the direct method (R =
> 0.95), averaging 191 against 222 bits/s, the direct method being higher in all sessions but one
> by 31 bits/s on average, so the burst code keeps about 85% of the information.

Direct method on binary words of 0.4 ms bins, words starting at every bin over 200 to 1000 ms,
entropies NSB-corrected (`ia1.nsb_entropy`). Readings (a) the rate at 8 bins (3.2 ms), (b) a
linear extrapolation in 1 / word length over 1 to 8 bins. The burst-code rate is I' of
fig7gh (non-bursting sessions as all 1-bursts), with bins (a) 0.1 ms and (b) 0.4 ms. Sessions (a)
all kept, (b) bursting.

In [ ]:
rows = []
for s in KEPT_A:
    rates = [ia1.direct_information(s.trials, L) for L in range(1, 9)]
    sl, ic, *_ = stats.linregress(1 / np.arange(1, 9), rates)
    row = dict(session=s.name, kept_c=s.name in POP["c"], bursting=LIM[s.name] > 0, direct_8=rates[-1], direct_extrap=ic)
    for bw in [1e-4, 4e-4]:
        row[f"burst_{bw * 1e3:.1f}"] = sum(v["rate"] for v in info[(s.name, bw)].values())
    rows.append(row)
DM = pd.DataFrame(rows)
summ = []
for dcol in ["direct_8", "direct_extrap"]:
    for bcol in ["burst_0.1", "burst_0.4"]:
        for lab, g in [("(a) kept (c)", DM[DM.kept_c]), ("(b) bursting (c)", DM[DM.kept_c & DM.bursting]),
                       ("(a) kept (a)", DM), ("(b) bursting (a)", DM[DM.bursting])]:
            diff = g[dcol] - g[bcol]
            summ.append(dict(direct=dcol, burst=bcol, sessions=lab, n=len(g), R=round(np.corrcoef(g[dcol], g[bcol])[0, 1], 3),
                             mean_direct=round(g[dcol].mean(), 1), sd_direct=round(g[dcol].std(ddof=1), 1),
                             mean_burst=round(g[bcol].mean(), 1), sd_burst=round(g[bcol].std(ddof=1), 1),
                             mean_diff=round(diff.mean(), 1), sd_diff=round(diff.std(ddof=1), 1),
                             direct_higher=f"{int((diff > 0).sum())}/{len(g)}",
                             lost_pct=round(100 * (1 - g[bcol].mean() / g[dcol].mean()), 1)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: partially reproduced.** With 0.4 ms bins for the burst code, over the 89 kept sessions
of selection (c), R = 0.967, the direct method gives 198.4 bits/s (8 bins) and the burst code
178.1, a mean difference of 20.3 bits/s, all within tolerance, and the burst code keeps 90% (10.2%
lost, outside 14 +/- 1.4). The direct method is higher in 74 of 89 sessions (83%), not in all but
one. Over the bursting sessions R = 0.951, 220.7 against 199.8 bits/s, difference 21.0, higher in
52 of 64. Over selection (a) the values are alike (R = 0.965, 204.3 against 183.6, 10.1% lost, 94
of 115). The extrapolated direct rate changes these by under 1 bit/s. With 0.1 ms bins the burst
code is above the direct method on average (220.1 against 198.4 bits/s).

## Burst-triggered averages: per-session measures

For each bursting session (main reading), the stimulus in dB (stimulus file shifted to the
intensity in `cell.dat`, as `stimtodB.m` does) around every burst onset in 200 to 1000 ms, from
-25 to +15 ms at 20 kHz (Methods, Estimation of burst-triggered averages). tau_n is the distance
from the nBTA's central maximum to the onset (Eq. 13 and the sentence after it: "The time tau_n
between the maximum of nBTA and tau = 0"), which the Results call "the mean delay between stimulus
upstroke and burst generation" (Qualitative description, paragraph 3). The central maximum is
therefore searched (a) before the onset, -25 to 0 ms, the main reading since revision 1; (b) over
the whole window, -25 to +15 ms, the reading of the first round, shown as a variant.

A session enters the analyses of n = 1 to 4 when it has at least two bursts of each n, the least
that gives a variance; the paper says only that eight sessions had "too few bursts" (Results,
paragraph 6), so this is an assumption, and the results with at least 20 and at least 50 bursts of
each n are shown too.

In [ ]:
def p_two_sided_ttests(a, b):
    return stats.ttest_ind(a, b, axis=0).pvalue


SEARCH = {"a": (-ia1.PRE, 0.0), "b": None}       # where the central maximum is searched
MIN_BURSTS = [2, 20, 50]
BTA = {}
for s in BURSTING_A:
    st = stim_db(s)
    tr, on, n = ia1.bursts(s.trials, LIM[s.name])
    seg = {}
    for m in range(1, 5):
        if (n == m).sum() >= 2:
            seg[m], tt = ia1.segments(st, on[n == m])
    sp = spikes_in(s)
    sta_seg, tt = ia1.segments(st, sp)
    BTA[s.name] = dict(seg=seg, tt=tt, sta_seg=sta_seg, on=on, n=n, tr=tr, mean_stim=st[4000:20000].mean(),
                       counts=[len(seg[m]) if m in seg else int((n == m).sum()) for m in range(1, 5)])
have4 = [s for s in BURSTING_A if len(BTA[s.name]["seg"]) == 4]
for kr in "ca":
    bs = [s for s in BURSTING_A if s.name in POP[kr]]
    print(f"kept ({kr}): bursting sessions {len(bs)}; with at least k bursts of each n = 1 to 4:",
          {k: sum(min(BTA[s.name]["counts"]) >= k for s in bs) for k in MIN_BURSTS})
cnt4 = np.array([BTA[s.name]["counts"][3] for s in have4 if s.name in POP["c"]])
print("kept (c), sessions with n = 1 to 4: number of 4-bursts, quantiles 0, 0.25, 0.5, 0.75, 1:", np.quantile(cnt4, [0, .25, .5, .75, 1]))


def subset(rows_df, kr, k):
    # Rows of sessions in selection kr with at least k bursts of each n = 1 to 4.
    ok = {s.name for s in have4 if s.name in POP[kr] and min(BTA[s.name]["counts"]) >= k}
    return rows_df[rows_df.session.isin(ok)]

## fig8-nbta-differ

> In every bursting cell the burst-triggered averages for n = 1 to 4 differ significantly (in the
> example cell their central peaks grow higher and wider with n); 50 of the 58 bursting sessions
> show significantly different central peaks in the time domain, the other eight having too few
> bursts.

Two-way ANOVAs (frequency bin x n) on the real and the imaginary parts of the Fourier transform of
each segment, each at 0.01 / 2; frequency bins (a) all, (b) up to the cutoff. Then at each time
point, t-tests of each nBTA against the three others, p < 0.01; a point is marked for n when all
three are significant. Central peak: from the minimum before to the minimum after the central
maximum, searched (a) before the onset or (b) over the whole window. Readings (a) at least one
marked point in the central peak of every n, (b) more than half of the points of every central
peak marked. The two-way ANOVAs treat every segment and frequency bin as an observation, as the
Methods describe; with hundreds to thousands of segments they reject in nearly every session.

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    tt = B["tt"]
    segs = [B["seg"][m] for m in range(1, 5)]
    ffts = [np.fft.rfft(g, axis=1) for g in segs]
    freqs = np.fft.rfftfreq(len(tt), 1 / ia1.STIM_RATE)
    row = dict(session=s.name, protocol=s.protocol, counts=[len(g) for g in segs])
    for fr, sel in [("a", slice(None)), ("b", freqs <= s.cutoff)]:
        ps = []
        for part in [np.real, np.imag]:
            gs = [part(f)[:, sel] for f in ffts]
            if part is np.imag:
                gs = [g[:, 1:] for g in gs]          # the imaginary part of the 0 Hz bin is 0
            ps.append(min(ia1.two_way_anova(gs)))
        row[f"anova_{fr}"] = min(ps) < 0.01 / 2
    btas = [g.mean(0) for g in segs]
    sig = []
    for i in range(4):
        x = np.ones(len(tt), bool)
        for j in range(4):
            if j != i:
                x &= p_two_sided_ttests(segs[i], segs[j]) < 0.01
        sig.append(x)
    for sr in "ab":
        marked = []
        for i in range(4):
            ip, a, b = ia1.central_peak(btas[i], tt, SEARCH[sr])
            marked.append(sig[i][a:b + 1].mean())
        row[f"peak_any_{sr}"] = all(m > 0 for m in marked)
        row[f"peak_most_{sr}"] = all(m > 0.5 for m in marked)
        row[f"marked_{sr}"] = [round(m, 2) for m in marked]
    rows.append(row)
N8 = pd.DataFrame(rows)
print(subset(N8, "c", 2).to_string(index=False))
out = []
for kr, k, sr in itertools.product("ca", MIN_BURSTS, "ab"):
    g = subset(N8, kr, k)
    out.append(dict(kept=kr, min_bursts=k, search=sr, sessions=len(g), bursting=sum(s.name in POP[kr] for s in BURSTING_A),
                    anova_a=int(g.anova_a.sum()), anova_b=int(g.anova_b.sum()),
                    peak_any=int((g.anova_a & g[f"peak_any_{sr}"]).sum()), peak_most=int((g.anova_a & g[f"peak_most_{sr}"]).sum())))
print()
print(pd.DataFrame(out).to_string(index=False))

**Result: partially reproduced.** Of the 64 bursting sessions of selection (c) (58 +/- 5.8 in the
paper, so just outside), 53 have at least two bursts of each n = 1 to 4, and all 53 pass the
frequency-domain ANOVA, with all frequency bins and with bins up to the cutoff. In the time domain,
with the central maximum searched before the onset (main reading), 38 of them have marked points
in every central peak (reading a), outside 50 +/- 5, and 5 have most of every central peak marked
(reading b); with the maximum searched over the whole window, 42 and 7. With at least 20 bursts of
each n, 39 sessions, 33 and 35; with at least 50, 28 sessions, 25 and 26. Over selection (a), 71
sessions all pass the ANOVA and 54 (60 over the whole window) have marked points in every central
peak. The ANOVA part of the claim holds; the time-domain count does not.

## fig8-latency-decreases

> The latency between the stimulus upstroke and burst onset decreases with n, so that high or wide
> deflections produce prompt responses with high-n bursts.

tau_n per session with bursts of n = 1 to 4. Readings (a) tau_n decreasing from n = 1 to 4 (tau_4 <
tau_1 with no increase between consecutive n) in the majority of those sessions; (b) the mean over
sessions of tau_n decreasing from 1 to 4.

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    row = dict(session=s.name, protocol=s.protocol)
    for sr in "ab":
        taus = [-B["tt"][ia1.central_peak(B["seg"][m].mean(0), B["tt"], SEARCH[sr])[0]] * 1e3 for m in range(1, 5)]
        row.update({f"tau{m}_{sr}": t for m, t in zip(range(1, 5), taus)})
        row[f"decreasing_{sr}"] = all(np.diff(taus) <= 0) and taus[3] < taus[0]
        row[f"tau4_below_tau1_{sr}"] = taus[3] < taus[0]
    rows.append(row)
LT = pd.DataFrame(rows)
print(subset(LT, "c", 2).round(2).to_string(index=False))
out = []
for kr, k, sr in itertools.product("ca", MIN_BURSTS, "ab"):
    g = subset(LT, kr, k)
    tau = g[[f"tau{m}_{sr}" for m in range(1, 5)]]
    out.append(dict(kept=kr, min_bursts=k, search=sr, sessions=len(g), decreasing=int(g[f"decreasing_{sr}"].sum()),
                    tau4_below_tau1=int(g[f"tau4_below_tau1_{sr}"].sum()), mean_tau_ms=tau.mean().round(2).tolist(),
                    negative_tau=int((tau.values < 0).any(axis=1).sum()),
                    wilcoxon_p=stats.wilcoxon(g[f"tau1_{sr}"], g[f"tau4_{sr}"]).pvalue if len(g) > 5 else np.nan))
print()
print(pd.DataFrame(out).to_string(index=False))

**Result: reproduced** (reading b). With the central maximum searched before the burst onset, the
mean latency over the 53 sessions of selection (c) falls with n: 5.80, 4.23, 3.32 and 2.67 ms for
n = 1 to 4, with no negative tau_n; 51 of 53 sessions have tau_4 < tau_1 (Wilcoxon p = 4.5e-9), but
only 20 of 53 decrease at every step (reading a, not a majority). With the maximum searched over
the whole window, as in the first round, the means are 5.80, 4.22, 3.14 and -0.50 ms, with a
negative tau_n in 33 sessions, where the 4BTA peaks after the onset. With at least 20 or 50 bursts
of each n the means also fall (6.01 to 2.12 and 5.88 to 2.05 ms), and over selection (a) they fall
from 5.75 to 2.46 ms.

## fig8e-sigma-minimum

> The standard deviation of the stimuli preceding an n-burst has a minimum at a position nearly the
> same for all n; over the bursting cells the SD of the minimum's position across n = 1 to 4
> averages 0.4 ms (SD 0.7 ms), roughly 0.05 times the inverse cutoff frequency.

sigma_n(tau) by Eq. 14. Window for the minimum (a) -25 to 0 ms, (b) -25 to +15 ms. The SD across
the four n is the sample SD (N - 1), as MATLAB's `std`; the population SD (N) is shown too. With
few segments sigma_n is noisy and its minimum wanders, so the result is shown for sessions with
at least 2, 20 and 50 bursts of each n.

In [ ]:
rows = []
for s in have4:
    B = BTA[s.name]
    tt = B["tt"]
    row = dict(session=s.name, cutoff=s.cutoff)
    for w, sel in [("a", tt <= 0), ("b", tt <= tt.max())]:
        pos = [tt[sel][np.argmin(B["seg"][m].std(0, ddof=1)[sel])] * 1e3 for m in range(1, 5)]
        row[f"pos_{w}"] = pos
        row[f"sd_{w}"] = np.std(pos, ddof=1)
        row[f"sd0_{w}"] = np.std(pos)
    rows.append(row)
SG = pd.DataFrame(rows)
out = []
for kr, k, w in itertools.product("ca", MIN_BURSTS, "ab"):
    g = subset(SG, kr, k)
    for col in [f"sd_{w}", f"sd0_{w}"]:
        out.append(dict(kept=kr, min_bursts=k, window=w, sd="N-1" if col.startswith("sd_") else "N", sessions=len(g),
                        mean_sd_ms=round(g[col].mean(), 3), sd_of_sd_ms=round(g[col].std(ddof=1), 3), median_ms=round(g[col].median(), 3),
                        times_cutoff=round((g[col] * g.cutoff * 1e-3).mean(), 4)))
print(pd.DataFrame(out).to_string(index=False))
print("\nkept (c), at least 2 bursts: median position of the minimum (ms), window (a), per n:",
      np.median(np.array(subset(SG, "c", 2).pos_a.tolist()), 0))

**Result: not reproduced.** The minimum of sigma_n lies near -6 ms for every n (medians -6.85,
-6.3, -4.8 and -6.6 ms over the 53 sessions of selection (c)), but the SD of its position across
n = 1 to 4 averages 2.37 ms (SD 2.44) in window (a) and 2.84 ms in window (b), outside 0.4 +/- 0.7,
and 2.05 and 2.46 ms with the population SD; times the cutoff it is 0.35 to 0.48, against 0.05.
Requiring more bursts of each n narrows it but not to the paper's value: 2.14 ms with at least 20
(39 sessions) and 1.74 ms with at least 50 (28 sessions; 0.19 times the cutoff), and 1.46 ms over
the 38 such sessions of selection (a).

## sigma-n-below-sta-sigma

> In 98% of bursting cells, for all n, the SD of the stimuli preceding an n-burst is smaller than
> the SD of the stimuli preceding all spikes, by a ratio of minima of 1.62 on average (SD 0.56).

"Smaller" read as (a) the minimum of sigma_n below the minimum of sigma, (b) sigma_n below sigma at
every tau in -25 to +15 ms; n = 1 to 4. The ratio is min sigma over the mean over n of min sigma_n,
minima over -25 to +15 ms (and over -25 to 0 ms as a variant).

The minimum over 800 time points of an SD estimated from few segments is biased low, so a smaller
minimum for the n-bursts could come from their smaller number alone. As a control (not in the
paper), sigma is also computed from random subsets of the segments preceding all spikes, of the
same size as each n's, 20 draws per n, and the minimum of sigma_n is compared with the mean
minimum of these size-matched sigmas.

In [ ]:
rng8 = np.random.default_rng(14)
rows = []
for s in have4:
    B = BTA[s.name]
    tt = B["tt"]
    sig = B["sta_seg"].std(0, ddof=1)
    sn = [B["seg"][m].std(0, ddof=1) for m in range(1, 5)]
    matched = [np.mean([B["sta_seg"][rng8.choice(len(B["sta_seg"]), len(B["seg"][m]), replace=False)].std(0, ddof=1).min()
                        for _ in range(20)]) for m in range(1, 5)]
    rows.append(dict(session=s.name, a=all(x.min() < sig.min() for x in sn), b=all((x < sig).all() for x in sn),
                     ratio=sig.min() / np.mean([x.min() for x in sn]),
                     ratio_pre=sig[tt <= 0].min() / np.mean([x[tt <= 0].min() for x in sn]),
                     **{f"ratio_n{m}": sig.min() / x.min() for m, x in zip(range(1, 5), sn)},
                     a_matched=all(x.min() < mm for x, mm in zip(sn, matched)),
                     ratio_matched=np.mean(matched) / np.mean([x.min() for x in sn]),
                     matched_ratio_of_sta=sig.min() / np.mean(matched)))
SB = pd.DataFrame(rows)
out = []
for kr, k in itertools.product("ca", MIN_BURSTS):
    g = subset(SB, kr, k)
    out.append(dict(kept=kr, min_bursts=k, sessions=len(g), a_pct=round(100 * g.a.mean(), 1), b_pct=round(100 * g.b.mean(), 1),
                    ratio=round(g.ratio.mean(), 3), ratio_sd=round(g.ratio.std(ddof=1), 3), ratio_pre=round(g.ratio_pre.mean(), 3),
                    **{f"ratio_n{m}": round(g[f"ratio_n{m}"].mean(), 3) for m in range(1, 5)},
                    a_vs_matched_pct=round(100 * g.a_matched.mean(), 1), ratio_vs_matched=round(g.ratio_matched.mean(), 3),
                    sta_vs_matched=round(g.matched_ratio_of_sta.mean(), 3)))
print(pd.DataFrame(out).to_string(index=False))

**Result: reproduced.** In 98.1% of the 53 sessions of selection (c) the minimum of sigma_n is
below the minimum of sigma for all n = 1 to 4 (reading a), within 98 +/- 9.8, and the ratio of the
minima averages 1.596 (SD 0.280), within 1.62 +/- 0.56. Read as sigma_n below sigma at every tau
(b), it holds in 1.9%. The result is not an effect of the smaller number of n-bursts: sigma from
random subsets of the spike-triggered segments of the same size has a minimum about 5% below that
of all segments (0.5 to 0.8% with at least 20 or 50 bursts of each n), the minimum of sigma_n is
below that size-matched minimum for every n in 94.3% of sessions, and the ratio to it is 1.517.
With at least 20 or 50 bursts of each n the values are 97.4% and 1.562, and 96.4% and 1.571. The
ratios per n are 1.27, 1.45 and 1.62 for n = 1 to 3; for n = 4 the mean is 23.5, inflated by
sessions with few 4-bursts, and 2.38 with at least 20.

## fig9-nbta-not-concatenated-1bta

> The nBTA differs from n 1BTAs shifted by the burst's own ISIs and summed; for a large fraction of
> cells the difference is significant, the fraction diminishing as n grows; results are similar
> with shifted STAs.

f_n(t) = sum over the burst's spikes of 1BTA(t - t_i) (Eq. 15). Summing n copies of a stimulus in
dB only compares with the nBTA once the mean stimulus is subtracted (as the paper does for Figure
9C); this is taken as the reading of Eq. 15, an assumption, and the average of the shifted copies
(the sum over n) is shown as a variant. The 1BTA (or STA) is computed from -60 to +15 ms so that
its shifts cover the window. Comparison over the interval between the minima flanking the central
maximum of the nBTA (searched before the onset): two-way ANOVAs (frequency bin x group) on the
real and imaginary parts of the Fourier transform, at 0.01 / 2, with observations (a) each burst's segment against its own f_n
(paired: ANOVA on the differences), (b) the segments and the f_n as two groups. Then t-tests at
each time point (p < 0.01) and sessions with more than 70% of the interval significant.

In [ ]:
def paired_anova(d):
    # Differences segment - f_n per burst (rows) and frequency bin (columns): p of a mean
    # difference (group main effect) and of its variation across bins (the interaction).
    Nb, F = d.shape
    m = d.mean()
    mf = d.mean(0)
    ss_err = ((d - mf) ** 2).sum()
    df_err = (Nb - 1) * F
    ms = ss_err / df_err
    p_main = stats.f.sf(Nb * F * m ** 2 / ms, 1, df_err)
    p_int = stats.f.sf(Nb * ((mf - m) ** 2).sum() / (F - 1) / ms, F - 1, df_err)
    return min(p_main, p_int)


rows = []
for s in BURSTING_A:
    B = BTA[s.name]
    st = stim_db(s)
    ms = B["mean_stim"]
    tr, on, n = B["tr"], B["on"], B["n"]
    long_t0 = 0.06
    one, tl = ia1.segments(st, on[n == 1], pre=long_t0)
    allsp, _ = ia1.segments(st, spikes_in(s), pre=long_t0)
    kernels = {"1BTA": one.mean(0) - ms, "STA": allsp.mean(0) - ms}
    i_zero = int(round(long_t0 * ia1.STIM_RATE))
    for m in [2, 3, 4]:
        if m not in B["seg"]:
            continue
        idx = np.flatnonzero(n == m)
        seg = B["seg"][m] - ms
        tt = B["tt"]
        ip, a, b = ia1.central_peak(seg.mean(0), tt, SEARCH["a"])
        if b - a < 3:
            continue
        # spike offsets of each burst relative to its onset
        trial_spikes = s.trials
        for kname, ker in kernels.items():
            for scale in ["sum", "mean"]:
                fn = np.zeros_like(seg)
                for r, k in enumerate(idx):
                    t = trial_spikes[tr[k]]
                    j = np.searchsorted(t, on[k])
                    offs = t[j:j + m] - on[k]
                    for o in offs:
                        sh = int(round(o * ia1.STIM_RATE))
                        fn[r] += ker[i_zero - 500 - sh:i_zero + 300 - sh]
                    if scale == "mean":
                        fn[r] /= m
                x, y = seg[:, a:b + 1], fn[:, a:b + 1]
                X, Y = np.fft.rfft(x, axis=1), np.fft.rfft(y, axis=1)
                pa = min(paired_anova(np.real(X - Y)), paired_anova(np.imag(X - Y)[:, 1:]) if X.shape[1] > 2 else 1)
                pb = min(min(ia1.two_way_anova([np.real(X), np.real(Y)])), min(ia1.two_way_anova([np.imag(X)[:, 1:], np.imag(Y)[:, 1:]])) if X.shape[1] > 2 else 1)
                tp = stats.ttest_ind(x, y, axis=0).pvalue
                rows.append(dict(session=s.name, kept_c=s.name in POP["c"], n=m, kernel=kname, scale=scale, bursts=len(idx), anova_a=pa < 0.005, anova_b=pb < 0.005,
                                 t_frac=(tp < 0.01).mean()))
N9 = pd.DataFrame(rows)
summ = []
for (kname, scale, m), g0 in N9.groupby(["kernel", "scale", "n"]):
    for kr, g in [("c", g0[g0.kept_c]), ("a", g0)]:
        summ.append(dict(kept=kr, kernel=kname, scale=scale, n=m, sessions=len(g), anova_a_pct=round(100 * g.anova_a.mean(), 1),
                         anova_b_pct=round(100 * g.anova_b.mean(), 1),
                         t70_a_pct=round(100 * (g.anova_a & (g.t_frac > 0.7)).mean(), 1),
                         t70_b_pct=round(100 * (g.anova_b & (g.t_frac > 0.7)).mean(), 1)))
print(pd.DataFrame(summ).to_string(index=False))

**Result: reproduced** (resting on subtracting the mean stimulus before summing the 1BTAs). Over the
bursting sessions of selection (c) the nBTA differs from the summed 1BTAs by the ANOVA in 100% of
the sessions for n = 2, 3 and 4 with paired observations (a), and in 100%, 100% and 98.1% with
grouped ones (b), so the fraction does not grow with n; the share with more than 70% of the
interval significant falls from 93.8% (n = 2) to 84.1% and 52.8% (n = 4). With the STA the ANOVA
fractions are 100%, 100% and 98.1% (a) and the 70% share falls from 100% to 37.7%; with the
average of the shifted copies in place of their sum the results are alike, and so are those over
selection (a). An ANOVA that rejects in nearly every session separates little; the criterion
specifies it.

## fig9c-population-nbta

> In the seven cells driven at 100 Hz cutoff and 6 dB, the population-average nBTAs show that
> bursts with more spikes follow higher or wider stimulus deflections, with large spread across
> cells.

Each session's 1BTA, 2BTA and 3BTA minus its mean stimulus, shifted so that its central maximum
sits at the mean tau_n over the sessions (shifting by tau_n, averaging and shifting back), averaged
over the bursting protocol-5 sessions with at least two bursts of n = 1 to 3. Width (a) full width
at half height, (b) between the minima flanking the central maximum. The central maximum is
searched before the onset and, as a variant, over the whole window.

In [ ]:
rows = []
fig, ax = plt.subplots(figsize=(6, 4))
for kr, sr in itertools.product("ca", "ab"):
    p5 = [s for s in BURSTING_A if s.name in POP[kr] and s.protocol == 5 and all(m in BTA[s.name]["seg"] for m in (1, 2, 3))]
    if sr == "a":
        print(f"kept ({kr}): protocol-5 sessions kept", sum(s.protocol == 5 and s.name in POP[kr] for s in KEPT_A),
              "; bursting with n = 1 to 3:", len(p5), [s.name for s in p5])
    for m in (1, 2, 3):
        tt = BTA[p5[0].name]["tt"]
        btas, peaks = [], []
        for s in p5:
            b = BTA[s.name]["seg"][m].mean(0) - BTA[s.name]["mean_stim"]
            btas.append(b)
            peaks.append(ia1.central_peak(b, tt, SEARCH[sr])[0])
        target = int(round(np.mean(peaks)))
        aligned = np.array([np.roll(b, target - p) for b, p in zip(btas, peaks)])
        avg = aligned.mean(0)
        ip, a, b2 = ia1.central_peak(avg, tt, SEARCH[sr])
        rows.append(dict(kept=kr, search=sr, sessions=len(p5), n=m, height_dB=avg[ip], fwhm_ms=ia1.fwhm(avg, ip, 0.0) / 20,
                         minima_width_ms=(b2 - a) / 20, spread_at_peak_dB=aligned[:, ip].std(ddof=1), tau_ms=-tt[ip] * 1e3))
        if kr == "c" and sr == "a":
            ax.plot(tt * 1e3, avg, label=f"{m}BTA")
            ax.fill_between(tt * 1e3, avg - aligned.std(0, ddof=1), avg + aligned.std(0, ddof=1), alpha=0.2)
ax.set(xlabel="time from burst onset (ms)", ylabel="stimulus minus its mean (dB)", title="kept (c), maximum searched before onset")
ax.legend()
plt.show()
F9C = pd.DataFrame(rows)
print(F9C.round(2).to_string(index=False))

**Result: reproduced.** Selection (c) holds 7 bursting protocol-5 sessions with bursts of n = 1 to
3, the paper's seven cells. Their population-average central peak is 3.13, 7.03 and 8.46 dB above
the mean stimulus for n = 1, 2 and 3, and its width grows from 5.00 to 5.50 to 7.35 ms at half
height (12.55, 13.60, 17.80 ms between minima). The SD across sessions at the peak is 0.54, 0.83
and 1.51 dB. The two searches for the maximum give the same values, and the 10 sessions of
selection (a) give 3.22, 7.11 and 8.60 dB.

## fig10-burst-n-vs-height

> The probability of a burst with n spikes after a stimulus maximum depends on its height, low
> excursions giving no response or an isolated spike and high ones doublets, triplets or 4-bursts;
> all bursting cells show this.

Methods (Relating burst probabilities to the height of stimulus excursions), per trial: each
n-burst shifted back by tau_n (the maximum of the nBTA searched (a) before the onset, (b) over
the whole window); each local maximum of the stimulus in 200 to 1000 ms matched to the
burst whose shifted onset is closest within [t0 - T, t0 + T]; a burst matched to more than one
maximum kept by the nearest, the other maxima counting as no response. T is the width of the
central STA peak, (a) between its flanking minima, (b) full width at half height. P(n | h) in
windows of half-width 5% of the span of h, at 41 centres. Segregation (a) the h at which P(n | h)
peaks increases with n for n = 0 to the largest n with bursts (up to 4); (b) the mean of h
weighted by P(n | h) increases with n. Bursting sessions with bursts of n = 1 and 2 at least.

In [ ]:
from scipy.signal import argrelmax


def p_n_given_h(s, Twin, sr):
    B = BTA[s.name]
    st = stim_db(s)
    i0, i1 = int(ia1.T0 * ia1.STIM_RATE), int(ia1.T1 * ia1.STIM_RATE)
    mx = argrelmax(st[i0:i1])[0] + i0
    tmax, h = mx / ia1.STIM_RATE, st[mx]
    tt = B["tt"]
    tau = {m: -tt[ia1.central_peak(B["seg"][m].mean(0), tt, SEARCH[sr])[0]] for m in B["seg"]}
    resp = []
    for i in range(s.n_trials):
        k = B["tr"] == i
        on, n = B["on"][k], B["n"][k]
        sh = on - np.array([tau.get(m, tau[max(tau)]) for m in n])
        best = np.full(len(tmax), -1)
        for j, t0 in enumerate(tmax):
            d = np.abs(sh - t0)
            if len(d) and d.min() <= Twin:
                best[j] = np.argmin(d)
        r = np.zeros(len(tmax), int)
        for bi in np.unique(best[best >= 0]):
            js = np.flatnonzero(best == bi)
            jbest = js[np.argmin(np.abs(tmax[js] - sh[bi]))]
            r[jbest] = n[bi]
        resp.append(r)
    resp = np.concatenate(resp)
    matched = int((resp > 0).sum())
    hh = np.tile(h, s.n_trials)
    span = hh.max() - hh.min()
    centres = np.linspace(hh.min(), hh.max(), 41)
    nmax = min(4, resp.max())
    P = np.array([[np.mean(resp[np.abs(hh - c) <= 0.05 * span] == m) if (np.abs(hh - c) <= 0.05 * span).any() else np.nan
                   for c in centres] for m in range(nmax + 1)])
    occupancy = np.array([(np.abs(hh - c) <= 0.05 * span).sum() for c in centres])
    return centres, P, nmax, matched, np.bincount(np.minimum(resp, 4), minlength=5), occupancy


rows = []
example = None
for s, sr in itertools.product(BURSTING_A, "ab"):
    B = BTA[s.name]
    if not (1 in B["seg"] and 2 in B["seg"]):
        continue
    tt = B["tt"]
    sta = B["sta_seg"].mean(0) - B["mean_stim"]
    ip, a, b = ia1.central_peak(sta, tt, SEARCH[sr])
    for tw, Tw in [("a", (b - a) / ia1.STIM_RATE), ("b", ia1.fwhm(sta, ip, 0.0) / ia1.STIM_RATE)]:
        centres, Pm, nmax, matched, nresp, occ = p_n_given_h(s, Tw, sr)
        peak_h = [centres[np.nanargmax(p)] for p in Pm]
        mean_h = [np.nansum(centres * p) / np.nansum(p) for p in Pm]
        rows.append(dict(session=s.name, kept_c=s.name in POP["c"], search=sr, T=tw, T_ms=Tw * 1e3, nmax=nmax, matched=matched, peak_h=np.round(peak_h, 1),
                         mean_h=np.round(mean_h, 1), n_resp=nresp.tolist(), peak_at_top=[int(np.nanargmax(p) == len(centres) - 1) for p in Pm], top_occupancy=int(occ[-1]), a=all(np.diff(peak_h) > 0), b=all(np.diff(mean_h) > 0),
                         a3=all(np.diff(peak_h[:4]) > 0), b3=all(np.diff(mean_h[:4]) > 0)))
        if example is None and nmax == 4 and tw == "a" and sr == "a" and s.name in POP["c"]:
            example = (s.name, centres, Pm)
H = pd.DataFrame(rows)
out = []
for kr, sr, tw in itertools.product("ca", "ab", "ab"):
    g = H[(H.kept_c | (kr == "a")) & (H.search == sr) & (H["T"] == tw)]
    out.append(dict(kept=kr, search=sr, T=tw, sessions=len(g), median_T_ms=round(g.T_ms.median(), 2), seg_a=int(g.a.sum()),
                    seg_a_pct=round(100 * g.a.mean(), 1), seg_b=int(g.b.sum()), seg_b_pct=round(100 * g.b.mean(), 1),
                    n0_3_a=int(g.a3.sum()), n0_3_b=int(g.b3.sum()), median_matched=int(g.matched.median()),
                    nmax_peak_at_top_pct=round(100 * np.mean([p[-1] for p in g.peak_at_top]), 1),
                    median_maxima_in_top_window=int(g.top_occupancy.median())))
print(pd.DataFrame(out).to_string(index=False))
side = H[H.kept_c & (H.search == "a")].pivot(index="session", columns="T", values=["matched", "a", "b", "nmax"])
print("\nkept (c), search (a): sessions where the two T readings match a different number of maxima:",
      int((side["matched"]["a"] != side["matched"]["b"]).sum()), "of", len(side), "; where flag (a) differs:",
      int((side["a"]["a"] != side["a"]["b"]).sum()), "; flag (b) differs:", int((side["b"]["a"] != side["b"]["b"]).sum()))
print(side.head(15).to_string())
ex = H[H.kept_c & (H.search == "a") & (H["T"] == "a")]
print("\nkept (c), search (a), T (a), first sessions: responses per n (0 to 4+), peak h and mean h per n")
print(ex[["session", "nmax", "n_resp", "peak_h", "mean_h", "peak_at_top", "top_occupancy"]].head(10).to_string(index=False))
name, centres, Pm = example
fig, ax = plt.subplots(figsize=(6, 4))
for m, p in enumerate(Pm):
    ax.plot(centres, p, label=f"n = {m}")
ax.set(xlabel="height of the stimulus maximum h (dB)", ylabel="P(n | h)", title=name, ylim=(0, 1))
ax.legend()
plt.show()

**Result: not reproduced.** Over the 64 bursting sessions of selection (c), with each n-burst
shifted back by tau_n found before the onset, the h at which P(n | h) peaks increases from n = 0 to
the largest n (up to 4) in 7 sessions (10.9%) and the P-weighted mean h in 15 (23.4%), under both
widths T, against all bursting cells. With tau_n searched over the whole window the counts are 7
and 16, and over selection (a) 8 and 19 to 21 of 84. Restricted to n = 0 to 3, a variant outside
the criterion, the ordering holds in 34 and 36 to 37 of 64 sessions: the 4-bursts, which are few,
most often break it. The two T readings match a different number of maxima in 58 of 64 sessions
(a median of 8316 against 8173), yet the peak flag changes in none and the mean flag in 2, one
each way: the peak of P(n | h) on the 41-point grid of h does not move when about 2% of the matches
change, which is why the counts are equal.

## firing-rate-no-effect-on-limit

> The firing rate has practically no effect on the limiting ISI; a 50 Hz increase in rate shifts it
> by less than 0.4 ms and raises the average n by less than 25%.

Reading (a), the same cell and stimulus at two target rates 50 Hz apart: the headers record target
rates of 100 and 90 Hz only (Data check), so no pair exists. Reading (b): linear regressions over
the kept sessions of the limiting ISI (bursting sessions) and of the average n on the session's
rate, scaled to 50 Hz; the increase of the average n is relative to its mean.

In [ ]:
rows = []
for s in KEPT_A:
    c = ia1.burst_counts(s.trials, LIM[s.name])
    rows.append(dict(session=s.name, kept_c=s.name in POP["c"], protocol=s.protocol, rate=D.loc[D.session == s.name, "rate"].iloc[0],
                     limit_ms=LIM[s.name] * 1e3, avg_n=ia1.average_n(c)[0]))
RT = pd.DataFrame(rows)
pairs = D[D.protocol.notna()].groupby(["cell", "stimulus"]).desired_rate.agg(lambda x: sorted(set(x.dropna())))
print("cell and stimulus with more than one target rate:", [k for k, v in pairs.items() if len(v) > 1])
for lab, g in [("kept (c)", RT[RT.kept_c]), ("kept (c), protocol 1 only", RT[RT.kept_c & (RT.protocol == 1)]),
               ("kept (a)", RT), ("kept (a), protocol 1 only", RT[RT.protocol == 1])]:
    gb = g[g.limit_ms > 0]
    r1 = stats.linregress(gb.rate, gb.limit_ms)
    r2 = stats.linregress(g.rate, g.avg_n)
    print(f"{lab}: limiting ISI slope {r1.slope:.4f} ms/Hz (p {r1.pvalue:.3g}, n {len(gb)}) -> shift for +50 Hz {50 * r1.slope:.3f} ms;"
          f" average n slope {r2.slope:.5f} per Hz (p {r2.pvalue:.3g}, n {len(g)}) -> +50 Hz changes it by {100 * 50 * r2.slope / g.avg_n.mean():.1f}%"
          f" of its mean {g.avg_n.mean():.3f}")

**Result: reproduced** (reading b). No cell was recorded at two target rates (the headers hold 100
and 90 Hz only), so reading (a) cannot be formed. Across the sessions of selection (c) a 50 Hz
higher rate goes with a limiting ISI shorter by 0.337 ms (p = 0.44, 64 bursting sessions), below
0.44 ms, and an average n higher by 15.7% (p = 0.132, 89 sessions), below 27.5%; within protocol 1
alone, 0.057 ms and 18.9%. Over selection (a), 0.189 ms and 13.6%. These are differences between
cells, not the within-cell change the paper may have measured.

## model-reproduces-coding-trends (blocked)

> The threshold-linear model with refractoriness reproduces the coding trends of the data, at
> single-cell and population level, though it is less precise in time than the real cells.

Depends on model-matches-classification, which is blocked (`data_missing`, no thresholds s_TH).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-sessions-and-trials | partially reproduced |
| table1-sessions-per-protocol | not reproduced |
| firing-rate-uniform | reproduced |
| threshold-population | blocked: data missing |
| table1-bursting-sessions | partially reproduced (assumed) |
| table1-isolated-spikes | not reproduced |
| table1-burst-length | not reproduced |
| fig5-burst-probability-vs-stimulus | reproduced (assumed) |
| fig3-correlation-shape | reproduced (assumed) |
| method-correlation-error | reproduced (assumed) |
| model-matches-classification | blocked: data missing |
| fig6-bursts-less-correlated | reproduced (assumed) |
| fig7ab-information-per-burst-increases | partially reproduced (assumed) |
| fig7cd-occurrence-decreases | partially reproduced (assumed) |
| fig7ef-jitter-independent-of-n | reproduced (assumed) |
| fig7gh-information-rate | not reproduced (assumed) |
| direct-method-comparison | partially reproduced (assumed) |
| fig8-nbta-differ | partially reproduced (assumed) |
| fig8-latency-decreases | reproduced (assumed) |
| fig8e-sigma-minimum | not reproduced (assumed) |
| sigma-n-below-sta-sigma | reproduced (assumed) |
| fig9-nbta-not-concatenated-1bta | reproduced (assumed) |
| fig9c-population-nbta | reproduced (assumed) |
| fig10-burst-n-vs-height | not reproduced (assumed) |
| firing-rate-no-effect-on-limit | reproduced (assumed) |
| model-reproduces-coding-trends | blocked: data missing |

How the analysis differs from the paper's:

- Population. The paper keeps 86 of its 132 sessions by the rate SD (< 35 Hz) and a rate near
  100 Hz. In the data set that rule keeps 115 of the 116 sessions of the paper's seven protocols.
  Leaving out also the 9 sessions with fewer than 98 trials (the paper gives 98 to 503) and the
  sessions sorted off-line (selection c) keeps 89, with 47, 6, 7, 7, 8, 7 and 7 per protocol
  against the paper's 43, 6, 7, 7, 8, 7 and 8, and bursting sessions per protocol equal to Table
  1's in six of seven. Selection (c) is the population from table1-bursting-sessions on, an
  assumption: the off-line exclusion has no passage in the paper behind it. The results over the
  115 sessions of selection (a), the population of the first round, are shown beside it.
- Burst identification. The Methods leave open which maximum is the first peak, which minimum
  bounds it and what sigma and N are in the error bar. The first local maximum (peak reading a)
  falls on edge effects of the refractory plateau, where C_s has no spike pairs, and leaves 11 of
  89 sessions bursting; started after the plateau (peak reading c) it gives 61, close to the 64 of
  the main reading. With N the number of trials the error bars are so wide that no session bursts.
  The main reading is the largest value below 5 ms, the first significant minimum after it, and N
  the number of product terms, under which the error bar is 2.6% of the span of C_s (the paper's
  3.4%). Outcomes that rest on bursts are marked as resting on this choice.
- Burst-triggered averages. The central maximum of an nBTA, which gives tau_n, is searched before
  the burst onset (the paper calls tau_n the delay between stimulus upstroke and burst
  generation); the first round searched the whole window, which put the 4BTA's maximum after the
  onset in many sessions. Sessions need at least two bursts of each n; results with at least 20
  and 50 are shown.
- Information. I_n(1) and I'_n are plug-in estimates, as the criterion gives them. A baseline with
  each trial's burst onsets shifted at random gives a larger rise of I_n(1) with n than the data,
  so that rise is estimator bias.
- Thresholds. The data set holds no threshold s_TH, so the model of Eq. 2 and the claims on it
  are blocked.
- Smaller choices are stated in each section: the stimulus is the AM signal in dB from
  `stimtodB.m` with the intensity of `cell.dat`; f_n of Eq. 15 sums 1BTAs after subtracting the
  mean stimulus; peaks of C_s are counted with a prominence threshold.

What was found about the dataset:

- Data1 holds 150 sessions (75 experiments of two stimuli each) of 45 cells; 116 sessions from
  37 cells on 23 dates use the paper's seven protocols, the paper's 37 cells; 28 sessions use the
  `gaussModality` and `lmd` stimuli and 6 use cutoffs of 50 and 1600 Hz, which the paper does not
  mention. Nothing marks the 86 sessions the paper kept.
- Data1 records no animal, no threshold and no best frequency; `cell.dat` gives the intensity
  played. 28 sessions were sorted off-line and have no header; in 25 of them the trial blocks run
  on into the pause (spikes up to 2.76 s, at a median 2.7 Hz from 1 to 1.7 s), while no on-line
  block goes past 1.02 s. Each block is one trial: none is sorted out of order or holds twice the
  usual spike count. `firing_rates` gives each
  spike file's trial count, which matches the blocks parsed here in 136 of 138 files, and a rate
  whose +/- value (median 128 Hz) is not the trial-to-trial SD (median 5.8 Hz).
- The intensity in three headers differs from `cell.dat` by more than 1 dB (up to 6.6 dB); the
  descriptor says 98 to 533 trials, the files hold 69 to 503; six sessions had pauses of 450 ms,
  not 700. Two experiments (03-01-15-aa, 03-06-27-ae) are missing from `singlecells.txt`.
- Spikes and stimuli share the trial clock: the STA peaks 5.6 ms (median) before the spike in
  every session and almost vanishes with the stimulus shifted by 100 ms.